# 01 — LAMP Structured Pruning dan Recovery Fine-tuning Biasa
Notebook standalone **YOLO26n dan YOLO26s**: baseline terlatih → pruning struktural LAMP →
evaluasi sebelum recovery → fine-tuning biasa → evaluasi setelah recovery. Tidak memakai
teacher, KD, sparse training, atau perubahan loss deteksi. Notebook 01/02/03 dipertahankan.

Default memproses extended pruning 10%, 20%, 30%, 40%, dan 50% channel eligible untuk N/S.
Setiap rasio memulai dari baseline asli, dengan satu budget recovery yang sama.
Pruning/reload yang valid tidak menjamin akurasi pulih; hasil negatif tetap disimpan.

Rujukan: [LAMP asli](https://arxiv.org/abs/2010.07611),
[Torch-Pruning 1.6.1](https://github.com/VainF/Torch-Pruning/tree/v1.6.1),
[Ultralytics Advanced Customization](https://docs.ultralytics.com/usage/engine/),
[Ultralytics Train](https://docs.ultralytics.com/modes/train/), dan
[MP-YOLO, Zhou dkk.](https://doi.org/10.1016/j.jvcir.2025.104560).
MP-YOLO memakai pruning dan fine-tuning; notebook ini merupakan adaptasi pada YOLO26,
bukan replikasi penuh arsitektur, stopping budget, maupun jadwal paper.

Kaggle: aktifkan GPU + Internet, tambahkan **Poltrada V3 YOLO26 Clean** baseline dan secret
`WANDB_API_KEY`, lalu Run All pada kernel baru. Sepuluh konfigurasi dapat memerlukan waktu panjang;
pilih satu model/rasio untuk uji pipeline.

## 02 — Konfigurasi Pruning dan Recovery
Semua parameter pengguna berada pada cell ini. Default N/S, extended, rasio 10–50% menghasilkan
sepuluh eksperimen independen. `MODEL_VARIANTS=["n"]` atau `["s"]` memilih satu model;
`PRUNING_RATIOS=[0.10]` membatasi uji. Baseline N memakai artifact `v84`, S memakai `v62`;
`best.pt` harus unik. Hash checkpoint lokal bersifat eksplisit per model.

Rasio adalah proporsi channel eligible, bukan seluruh parameter. Cap group tetap 50%, satu
langkah, tanpa rounding. Fine-tuning memakai optimizer baru (`resume=False`), AdamW dan
learning rate eksplisit, FP32 tanpa AMP, loss deteksi native Ultralytics, dan validation
untuk memilih best EMA. Anggaran `RECOVERY_EPOCHS` sama pada seluruh konfigurasi.
Tidak ada recovery kumulatif antar-rasio atau penggunaan hasil N sebagai baseline S.

In [ ]:
from pathlib import Path

MODEL_VARIANTS = ["n", "s"]
WANDB_ENTITY = "ilhamrafiqin-brawijaya-university"
WANDB_PROJECT = "yolom26-kd-indotraffic"
MODEL_ARTIFACTS = {
    "n": "ilhamrafiqin-brawijaya-university/yolom26-kd-indotraffic/03_baseline_yolo26n-9uu3meve-checkpoints:v84",
    "s": "ilhamrafiqin-brawijaya-university/yolom26-kd-indotraffic/02_baseline_yolo26s-q5kp9k9d-checkpoints:v62",
}
LOCAL_CHECKPOINT_PATHS = {"n": None, "s": None}
LOCAL_EXPECTED_SHA256S = {"n": None, "s": None}
CHECKPOINT_SELECTOR = "best.pt"
DATASET_ROOT = Path("/kaggle/input/datasets/ilhamrafiqin/poltrada-v3-yolo26-clean")
DATA_YAML = DATASET_ROOT / "data.yaml"
OUTPUT_DIR = Path("/kaggle/working/pruning/lamp_v2_finetune") if Path("/kaggle").exists() else Path("outputs/pruning/lamp_v2_finetune")
PRUNING_POLICY = "extended"
PRUNING_RATIOS = [0.10, 0.20, 0.30, 0.40, 0.50]
GLOBAL_PRUNING = True
ITERATIVE_STEPS = 1
MAX_PRUNING_RATIO = 0.50
ROUND_TO = None
IMPORTANCE_P = 2
SEED = 42
IMGSZ = 640
DEVICE = "auto"
RUN_FULL_VALIDATION = True
RUN_LATENCY_BENCHMARK = True
BATCH = 16
WORKERS = 2
LATENCY_WARMUP = 50
LATENCY_REPEATS = 200
RELOAD_TIMEOUT_SECONDS = 300
INSTALL_DEPENDENCIES = True
WANDB_TRACKING = True
UPLOAD_WANDB_ARTIFACTS = True
EDA_DPI = 300
EDA_MAX_LAYERS_PER_PAGE = 32
EDA_DISPLAY_RATIO = 0.50
RECOVERY_ENABLED = True
FINETUNING_ENABLED = True
KD_ENABLED = False

RECOVERY_EPOCHS = 5
RECOVERY_LR0 = 0.0001
RECOVERY_OPTIMIZER = "AdamW"
RECOVERY_PATIENCE = 0
RECOVERY_CLOSE_MOSAIC = 1
RECOVERY_WARMUP_EPOCHS = 3
RECOVERY_WEIGHT_DECAY = 0.0005
RECOVERY_BATCH = 16
RECOVERY_NBS = 64
RECOVERY_WORKERS = 2
RECOVERY_AMP = False

## 03 — Environment dan Dependencies
Pin Ultralytics 8.4.155, Torch-Pruning 1.6.1, dan W&B 0.28.1. Bootstrap mempertahankan
Torch/torchvision Kaggle serta berhenti saat dependency tidak kompatibel. Source trainer
versi terpasang menjadi kontrak implementasi; dokumentasi terbaru dapat memiliki API berbeda.
Pengukuran dilakukan pada salinan unfused FP32; prediction/validation memakai wrapper terpisah.

In [ ]:
import csv
import hashlib
import importlib.metadata as metadata
import inspect
import json
import math
import os
import platform
import random
import subprocess
import sys
import time
import traceback
import uuid
import warnings
from contextlib import contextmanager
from copy import deepcopy
from datetime import datetime, timezone
from types import SimpleNamespace

PINNED = {"ultralytics": "8.4.155", "wandb": "0.28.1", "torch-pruning": "1.6.1"}
REQUIRED_CHECKS = (
    "baseline_loaded", "baseline_forward", "dependency_graph", "structural_channels_removed",
    "parameters_reduced", "pruned_forward", "protected_head_unchanged", "checkpoint_saved",
    "checkpoint_reloaded", "reload_consistent", "detection_prediction",
)


def write_json(path, value):
    """Tulis JSON standar; NaN/Inf ditolak, bukan disamarkan sebagai metrik."""
    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")


def sha256_file(path):
    """Hash berkas tanpa membaca seluruh checkpoint ke RAM sekaligus."""
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

In [ ]:
# Backward-compatible N helper aliases; public inputs are the per-model mappings.
WANDB_ARTIFACT = MODEL_ARTIFACTS["n"]
LOCAL_CHECKPOINT_PATH = None
LOCAL_EXPECTED_SHA256 = None


def model_spec(model_variant="n"):
    if model_variant not in {"n", "s"}:
        raise ValueError("Model variant harus n atau s.")
    local = LOCAL_CHECKPOINT_PATHS.get(model_variant)
    expected_hash = LOCAL_EXPECTED_SHA256S.get(model_variant)
    if model_variant == "n" and LOCAL_CHECKPOINT_PATH is not None:
        local, expected_hash = LOCAL_CHECKPOINT_PATH, LOCAL_EXPECTED_SHA256
    return {"model_variant": model_variant, "model_name": f"YOLO26{model_variant}",
            "artifact": MODEL_ARTIFACTS[model_variant], "local_checkpoint": local,
            "expected_sha256": expected_hash}


def policies_to_run(policy):
    if policy not in {"conservative", "extended", "both"}:
        raise ValueError("PRUNING_POLICY harus conservative, extended, atau both.")
    return ["conservative", "extended"] if policy == "both" else [policy]


def configuration(policy=None, ratio=None, model_variant=None):
    keys = ("MODEL_VARIANTS MODEL_ARTIFACTS LOCAL_CHECKPOINT_PATHS LOCAL_EXPECTED_SHA256S CHECKPOINT_SELECTOR "
            "DATASET_ROOT DATA_YAML OUTPUT_DIR PRUNING_POLICY PRUNING_RATIOS GLOBAL_PRUNING "
            "ITERATIVE_STEPS MAX_PRUNING_RATIO ROUND_TO IMPORTANCE_P SEED IMGSZ DEVICE "
            "RUN_FULL_VALIDATION RUN_LATENCY_BENCHMARK BATCH WORKERS INSTALL_DEPENDENCIES "
            "WANDB_TRACKING UPLOAD_WANDB_ARTIFACTS LATENCY_WARMUP LATENCY_REPEATS "
            "RELOAD_TIMEOUT_SECONDS RECOVERY_ENABLED FINETUNING_ENABLED KD_ENABLED "
            "EDA_DPI EDA_MAX_LAYERS_PER_PAGE EDA_DISPLAY_RATIO").split()
    def serializable(value):
        if isinstance(value, Path):
            return str(value)
        if isinstance(value, dict):
            return {key: serializable(item) for key, item in value.items()}
        if isinstance(value, (tuple, list)):
            return [serializable(item) for item in value]
        return value
    result = {key: serializable(globals()[key]) for key in keys}
    if policy is not None:
        result["policy"] = policy
    if ratio is not None:
        result["ratio"] = ratio
    if model_variant is not None:
        result["selected_model"] = serializable(model_spec(model_variant))
    return result


def validate_configuration():
    if not MODEL_VARIANTS or len(set(MODEL_VARIANTS)) != len(MODEL_VARIANTS):
        raise ValueError("MODEL_VARIANTS harus nonkosong dan tanpa duplikat.")
    for model_variant in MODEL_VARIANTS:
        artifact = model_spec(model_variant)["artifact"]
        if artifact.count("/") != 2 or ":" not in artifact:
            raise ValueError("Artifact wajib entity/project/artifact:version.")
    policies_to_run(PRUNING_POLICY)
    if not PRUNING_RATIOS or len(set(PRUNING_RATIOS)) != len(PRUNING_RATIOS):
        raise ValueError("PRUNING_RATIOS harus nonkosong dan tanpa duplikat.")
    if any(isinstance(r, bool) or not isinstance(r, (int, float)) or not math.isfinite(r)
           or not 0 < r <= MAX_PRUNING_RATIO for r in PRUNING_RATIOS):
        raise ValueError("Setiap ratio wajib finite dan 0 < ratio <= MAX_PRUNING_RATIO.")
    if MAX_PRUNING_RATIO != 0.50 or ROUND_TO is not None or not GLOBAL_PRUNING:
        raise ValueError("V2 memakai global ranking, cap 0.50, dan ROUND_TO=None.")
    if ITERATIVE_STEPS != 1 or IMPORTANCE_P != 2:
        raise ValueError("V2 hanya mendukung one-shot dengan p=2.")
    if any((RECOVERY_ENABLED, FINETUNING_ENABLED, KD_ENABLED)):
        raise ValueError("Recovery, fine-tuning, dan KD wajib disabled.")
    if IMGSZ < 32 or IMGSZ % 32 or BATCH < 1 or WORKERS < 0:
        raise ValueError("IMGSZ kelipatan 32, BATCH positif, WORKERS nonnegatif.")
    if EDA_DPI < 72 or EDA_MAX_LAYERS_PER_PAGE < 1 or not 0 < EDA_DISPLAY_RATIO <= 0.5:
        raise ValueError("Konfigurasi EDA DPI/page/ratio invalid.")
    if LATENCY_WARMUP < 1 or LATENCY_REPEATS < 1 or RELOAD_TIMEOUT_SECONDS < 1:
        raise ValueError("Jumlah pengukuran dan timeout wajib positif.")


def write_csv(path, rows):
    if not rows:
        return
    keys = list(dict.fromkeys(key for row in rows for key in row))
    with Path(path).open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=keys)
        writer.writeheader()
        for row in rows:
            writer.writerow({key: json.dumps(value, ensure_ascii=False) if isinstance(value, (list, dict))
                             else value for key, value in row.items()})


def model_state_sha256(core):
    """Buktikan setiap konfigurasi dimulai dari bobot/buffer FP32 yang identik."""
    digest = hashlib.sha256()
    for name, value in sorted(core.state_dict().items()):
        tensor = value.detach().cpu().contiguous()
        digest.update(name.encode())
        digest.update(str((str(tensor.dtype), tuple(tensor.shape))).encode())
        digest.update(tensor.numpy().tobytes())
    return digest.hexdigest()


def metric_comparison(before, after):
    result = {}
    for key, label in (("parameters", "parameter"), ("gflops", "gflops"),
                       ("checkpoint_bytes", "checkpoint")):
        base, pruned = before.get(key), after.get(key)
        result[label + "_reduction_pct"] = 100 * (base - pruned) / base if base else None
    base = (before.get("validation") or {}).get("map50_95")
    pruned = (after.get("validation") or {}).get("map50_95")
    delta = pruned - base if base is not None and pruned is not None else None
    result.update({"delta_map50_95": delta, "delta_map50_95_pp": 100 * delta if delta is not None else None,
                   "relative_map50_95_pct": 100 * delta / base if delta is not None and base else None})
    return result


def lamp_v2_display_name(policy, ratio_tag, model_variant, recovery_epochs=None):
    model_spec(model_variant)
    policy_suffix = "_c" if policy == "conservative" else ""
    recovery_suffix = f"_ft{recovery_epochs}" if recovery_epochs is not None else ""
    return f"prun_lamp2_yolo26{model_variant}{policy_suffix}{recovery_suffix}_pruned-{ratio_tag}"


def run_identity(policy, ratio, suite_id, model_variant="n"):
    tag = f"pr{ratio:g}"
    return {"name": lamp_v2_display_name(policy, tag, model_variant),
            "id": uuid.uuid4().hex, "resume": "never", "group": f"lamp-v2-{suite_id}",
            "job_type": "lamp-structured-pruning-v2",
            "tags": [f"yolo26{model_variant}", "lamp", "v2", "structured", "pruned", policy,
                     "no-recovery", tag, f"seed{SEED}"]}


def safe_error(error):
    message = str(error)
    secret = os.environ.get("WANDB_API_KEY")
    return message.replace(secret, "[REDACTED]") if secret else message

In [ ]:
_pruning_configuration = configuration
_validate_pruning_configuration = validate_configuration


def recovery_settings():
    return {"epochs": RECOVERY_EPOCHS, "optimizer": RECOVERY_OPTIMIZER, "lr0": RECOVERY_LR0,
            "lrf": 0.01, "weight_decay": RECOVERY_WEIGHT_DECAY, "momentum": 0.937,
            "warmup_epochs": RECOVERY_WARMUP_EPOCHS, "warmup_bias_lr": 0.0,
            "close_mosaic": RECOVERY_CLOSE_MOSAIC, "patience": RECOVERY_PATIENCE,
            "batch": RECOVERY_BATCH, "nbs": RECOVERY_NBS, "workers": RECOVERY_WORKERS, "imgsz": IMGSZ,
            "device": str(TARGET_DEVICE) if "TARGET_DEVICE" in globals() else DEVICE,
            "seed": SEED, "deterministic": True, "amp": RECOVERY_AMP,
            "resume": False, "freeze": None, "cache": False, "save": True, "val": True,
            "plots": True, "cos_lr": False, "mosaic": 1.0, "mixup": 0.0,
            "hsv_h": 0.015, "hsv_s": 0.7, "hsv_v": 0.4, "degrees": 0.0,
            "translate": 0.1, "scale": 0.5, "shear": 0.0, "perspective": 0.0,
            "flipud": 0.0, "fliplr": 0.5, "multi_scale": False}


def configuration(policy=None, ratio=None, model_variant=None):
    result = _pruning_configuration(policy, ratio, model_variant)
    result.update({"experiment_type": "lamp_structured_pruning_then_ordinary_finetune",
                   "recovery": recovery_settings(), "checkpoint_selection": "best_validation_fitness_ema_fp32"})
    return result


def validate_configuration():
    if RECOVERY_ENABLED is not True or FINETUNING_ENABLED is not True or KD_ENABLED is not False:
        raise ValueError("Notebook recovery memerlukan fine-tuning aktif dan KD disabled.")
    # Reuse V2 pruning validation without changing its no-recovery contract or source.
    flags = {key: globals()[key] for key in ("RECOVERY_ENABLED", "FINETUNING_ENABLED", "KD_ENABLED")}
    try:
        globals().update(dict.fromkeys(flags, False))
        _validate_pruning_configuration()
    finally:
        globals().update(flags)
    if isinstance(RECOVERY_EPOCHS, bool) or not isinstance(RECOVERY_EPOCHS, int) or RECOVERY_EPOCHS < 1:
        raise ValueError("RECOVERY_EPOCHS wajib integer positif.")
    if RECOVERY_OPTIMIZER not in {"AdamW", "SGD"}:
        raise ValueError("Optimizer recovery wajib eksplisit AdamW atau SGD; auto tidak digunakan.")
    for value in (RECOVERY_LR0, RECOVERY_WEIGHT_DECAY, RECOVERY_WARMUP_EPOCHS):
        if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value) or value < 0:
            raise ValueError("Hyperparameter recovery wajib finite dan nonnegatif.")
    if RECOVERY_LR0 <= 0 or RECOVERY_BATCH < 1 or RECOVERY_WORKERS < 0 or RECOVERY_AMP is not False:
        raise ValueError("Recovery memakai LR/batch positif, workers nonnegatif dan FP32 amp=False.")
    if isinstance(RECOVERY_NBS, bool) or not isinstance(RECOVERY_NBS, int) or RECOVERY_NBS < 1:
        raise ValueError("RECOVERY_NBS wajib integer positif untuk batch efektif.")
    if RECOVERY_PATIENCE != 0 or not 0 <= RECOVERY_CLOSE_MOSAIC <= RECOVERY_EPOCHS:
        raise ValueError("Budget tetap memakai patience=0 dan close_mosaic <= recovery epochs.")

In [ ]:
def ensure_dependencies(output_dir, install=True):
    """Pasang dependency tambahan sambil mengunci Torch/torchvision environment."""
    if sys.version_info < (3, 11):
        raise RuntimeError("Gunakan Python >=3.11.")
    preserved = {name: metadata.version(name) for name in ("torch", "torchvision")}
    needed = []
    for name, expected in PINNED.items():
        try:
            found = metadata.version(name)
        except metadata.PackageNotFoundError:
            found = None
        if found != expected:
            needed.append(f"{name}=={expected}")
    if needed and not install:
        raise RuntimeError(f"Dependency belum cocok: {needed}; aktifkan INSTALL_DEPENDENCIES.")
    if needed:
        constraints = Path(output_dir) / "preserve_torch.txt"
        constraints.write_text("\n".join(f"{k}=={v}" for k, v in preserved.items()) + "\n", encoding="utf-8")
        subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
                        "--constraint", str(constraints), *needed], check=True)
    actual = {name: metadata.version(name) for name in preserved}
    if actual != preserved:
        raise RuntimeError(f"Torch/torchvision berubah: {preserved} -> {actual}; restart runtime asli.")
    for name, expected in PINNED.items():
        if metadata.version(name) != expected:
            raise RuntimeError(f"Versi {name} tidak sesuai pin {expected}; restart kernel bila perlu.")

In [ ]:
validate_configuration()
SUITE_DIR = Path(OUTPUT_DIR).expanduser().resolve() / (
    datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
)
SUITE_DIR.mkdir(parents=True, exist_ok=False)
(SUITE_DIR / "baseline").mkdir()
write_json(SUITE_DIR / "config.json", configuration())
try:
    ensure_dependencies(SUITE_DIR, install=INSTALL_DEPENDENCIES)
except Exception as error:
    write_json(SUITE_DIR / "environment_failure.json", {"status": "FAIL", "error": safe_error(error)})
    raise

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch_pruning as tp
import ultralytics
import wandb
import yaml
from PIL import Image
from ultralytics import YOLO
from ultralytics.data.utils import check_det_dataset
from ultralytics.nn.modules import Bottleneck, C2PSA, Detect
from ultralytics.nn.modules.block import AAttn, Attention
from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils.torch_utils import _attention_ops
from thop import profile as thop_profile

In [ ]:
def seed_environment(seed):
    """Samakan RNG dan backend FP32 di notebook dan proses restore."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.set_float32_matmul_precision("highest")


def environment_report():
    """Periksa API aktual, bukan API yang diasumsikan dari contoh YOLOv8."""
    for name, expected in PINNED.items():
        if metadata.version(name) != expected:
            raise RuntimeError(f"Pin {name} tidak terpenuhi.")
    if ultralytics.__version__ != PINNED["ultralytics"]:
        raise RuntimeError("Modul Ultralytics lama masih di kernel; restart runtime.")
    signature = inspect.signature(tp.pruner.BasePruner)
    for key in ("global_pruning", "max_pruning_ratio", "round_to", "output_transform", "forward_fn"):
        if key not in signature.parameters:
            raise RuntimeError(f"BasePruner tidak mendukung {key}.")
    return {
        "python": sys.version, "platform": platform.platform(),
        "packages": {name: metadata.version(name) for name in (*PINNED, "torch", "torchvision", "numpy", "ultralytics-thop")},
        "cuda_available": torch.cuda.is_available(), "cuda_build": torch.version.cuda,
        "device": str(TARGET_DEVICE),
        "gpu": torch.cuda.get_device_name(TARGET_DEVICE) if TARGET_DEVICE.type == "cuda" else None,
        "precision": "FP32", "tf32": False, "cudnn_benchmark": False,
        "base_pruner_signature": str(signature),
        "lamp_signature": str(inspect.signature(tp.importance.LAMPImportance)),
        "lamp_source_sha256": hashlib.sha256(inspect.getsource(tp.importance.LAMPImportance).encode()).hexdigest(),
        "importance_module_sha256": sha256_file(inspect.getfile(tp.importance.LAMPImportance)),
        "flops_method": "THOP full-resolution Conv/BN + separate Ultralytics attention matmul counter; GFLOPs=2*MACs/1e9",
        "thop_profile_signature": str(inspect.signature(thop_profile)),
        "thop_distribution_versions": thop_versions(),
    }


def thop_versions():
    """Catat kedua distribution yang dapat menyediakan namespace thop yang sama."""
    result = {}
    for name in ("thop", "ultralytics-thop"):
        try:
            result[name] = metadata.version(name)
        except metadata.PackageNotFoundError:
            result[name] = None
    return result

In [ ]:
TARGET_DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu") if DEVICE == "auto" else torch.device(DEVICE)
seed_environment(SEED)
try:
    ENVIRONMENT = environment_report()
    write_json(SUITE_DIR / "environment.json", ENVIRONMENT)
except Exception as error:
    write_json(SUITE_DIR / "environment_failure.json", {"status": "FAIL", "error": safe_error(error)})
    raise
print("Runtime:", TARGET_DEVICE, "FP32", ENVIRONMENT["packages"])

## 04 — Pengambilan Baseline dari W&B
Unduh artifact satu kali per model. UUID/nama run eksperimen berbeda dari baseline/KD; project tetap sama.
Salinan lokal memerlukan pilihan eksplisit dan tingkat verifikasi provenance yang jujur.
SHA-256 dan identitas artifact disimpan; jumlah kelas tidak diasumsikan dari nama project.

In [ ]:
def select_checkpoint(root, selector="best.pt"):
    """Pilih checkpoint unik secara eksplisit; tidak menebak dari urutan filesystem."""
    root = Path(root).expanduser().resolve()
    if root.is_file():
        candidates = [root] if root.suffix.lower() == ".pt" else []
        base = root.parent
    else:
        candidates = sorted(p.resolve() for p in root.rglob("*.pt") if p.is_file())
        base = root
    print("Kandidat checkpoint:", [p.relative_to(base).as_posix() for p in candidates])
    if not candidates:
        raise FileNotFoundError(f"Tidak ada file .pt di {root}.")
    if root.is_file():
        return root  # LOCAL_CHECKPOINT_PATH berupa file adalah pilihan eksplisit.
    if selector:
        selector = str(selector).replace("\\", "/")
        matches = [p for p in candidates if p.relative_to(base).as_posix() == selector] if "/" in selector else [p for p in candidates if p.name == selector]
        if len(matches) == 1:
            return matches[0]
        raise ValueError(f"Selector {selector!r} memiliki {len(matches)} kecocokan; pilih path relatif tepat.")
    if len(candidates) == 1 and candidates[0].name == "best.pt":
        return candidates[0]
    raise ValueError("Checkpoint ambigu atau best.pt tidak tersedia; isi CHECKPOINT_SELECTOR secara eksplisit.")


def checkpoint_provenance(path, artifact_name, *, expected_hash=None, resolved_artifact=None):
    """Catat identitas sumber; hash lokal opsional wajib cocok bila diberikan."""
    digest = sha256_file(path)
    if expected_hash is not None and digest.lower() != expected_hash.lower():
        raise ValueError("SHA-256 checkpoint lokal tidak cocok dengan LOCAL_EXPECTED_SHA256.")
    return {"requested_artifact": artifact_name, "resolved_artifact": resolved_artifact,
            "checkpoint_name": Path(path).name, "checkpoint_path": str(Path(path).resolve()),
            "sha256": digest, "file_bytes": Path(path).stat().st_size,
            "source_verification": "wandb_download" if resolved_artifact else ("expected_sha256_matched" if expected_hash else "user_asserted_local_copy")}

In [ ]:
def wandb_session_login():
    if "WANDB_API_KEY" not in os.environ and Path("/kaggle").exists() and not wandb.api.api_key:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret("WANDB_API_KEY")
        wandb.login(key=key, relogin=False)
        del key


def obtain_baseline(suite_dir, model_variant="n"):
    """Ambil sumber satu kali via API; setiap run eksperimen dibuat terpisah."""
    spec = model_spec(model_variant)
    if spec["local_checkpoint"] is not None:
        path = select_checkpoint(spec["local_checkpoint"], CHECKPOINT_SELECTOR)
        return path, {**checkpoint_provenance(path, spec["artifact"], expected_hash=spec["expected_sha256"]),
                      "model_variant": model_variant}
    if not WANDB_TRACKING:
        raise ValueError("Isi LOCAL_CHECKPOINT_PATH atau aktifkan WANDB_TRACKING untuk sumber W&B.")
    wandb_session_login()
    artifact = wandb.Api(timeout=30).artifact(spec["artifact"], type="model")
    root = artifact.download(root=str(Path(suite_dir) / "baseline" / "source_artifact"))
    path = select_checkpoint(root, CHECKPOINT_SELECTOR)
    provenance = checkpoint_provenance(path, spec["artifact"], resolved_artifact=artifact.qualified_name)
    provenance.update({"artifact_digest": artifact.digest, "artifact_id": artifact.id,
                       "artifact_version": artifact.version,
                       "artifact_relative_path": path.relative_to(Path(root).resolve()).as_posix(),
                       "model_variant": model_variant})
    return path, provenance

In [ ]:
display(pd.DataFrame([model_spec(variant) for variant in MODEL_VARIANTS]))
print("Pengambilan sumber dan verifikasi SHA-256 dilakukan terpisah pada setiap model saat eksekusi suite.")

## 05 — Audit Arsitektur YOLO26n/S
Validasi actual checkpoint: skala nano/stem 16 atau small/stem 32, class names/order, head one-to-many dan one-to-one,
residual, concat, CSP, attention, dan sumber fitur. Pengukuran GFLOPs memakai THOP Conv/BN
pada resolusi penuh ditambah counter matmul attention, dengan metode identik sebelum/sesudah.
Tabel layer memakai hitungan operasi forward Conv/BN; operasi attention terpisah pada total.

In [ ]:
def ordered_names(names):
    """Pertahankan identitas dan urutan indeks kelas."""
    if isinstance(names, dict):
        indexed = {int(k): str(v) for k, v in names.items()}
        if sorted(indexed) != list(range(len(indexed))):
            raise ValueError("Indeks kelas wajib berurutan mulai dari nol.")
        return [indexed[i] for i in range(len(indexed))]
    return [str(value) for value in names]


def assert_class_match(model_names, dataset_names):
    """Jangan remap kelas diam-diam untuk membuat evaluasi berjalan."""
    if ordered_names(model_names) != ordered_names(dataset_names):
        raise ValueError(f"Urutan/nama kelas berbeda: model={ordered_names(model_names)}, dataset={ordered_names(dataset_names)}")


def architecture_audit(core, model_variant="n"):
    """Bangun allowlist hidden channel berdasarkan tipe modul dan hubungan konsumen."""
    if not isinstance(core, DetectionModel):
        raise TypeError("Checkpoint bukan DetectionModel Ultralytics.")
    heads = [(name, module) for name, module in core.named_modules() if isinstance(module, Detect)]
    if len(heads) != 1:
        raise ValueError("Diperlukan tepat satu Detect head.")
    head_name, head = heads[0]
    cfg = core.yaml
    first_conv = next(module for module in core.modules() if isinstance(module, torch.nn.Conv2d))
    if "yolo26" not in str(cfg.get("yaml_file", "")).lower() or cfg.get("reg_max") != 1 or first_conv.out_channels != {"n": 16, "s": 32}[model_variant]:
        raise ValueError("Metadata/struktur tidak cocok dengan YOLO26 N/S standar; audit manual diperlukan.")
    if cfg.get("scale") not in (None, "", model_variant):
        raise ValueError("Checkpoint bukan skala model yang dipilih.")
    branches = ("cv2", "cv3", "one2one_cv2", "one2one_cv3")
    if any(getattr(head, attr, None) is None or len(getattr(head, attr)) != head.nl for attr in branches):
        raise ValueError("Kedua cabang head wajib tersedia; checkpoint fused/head berubah ditolak.")
    names = ordered_names(core.names)
    if len(names) != head.nc or head.no != head.nc + 4 * head.reg_max:
        raise ValueError("Metadata kelas/dimensi Detect tidak konsisten.")
    modules = dict(core.named_modules())
    attention_prefixes = [name for name, m in modules.items() if isinstance(m, (C2PSA, Attention, AAttn))]
    allowed = {}
    for name, module in modules.items():
        if not isinstance(module, Bottleneck) or name.startswith(head_name + "."):
            continue
        if any(name == prefix or name.startswith(prefix + ".") for prefix in attention_prefixes):
            continue
        producer, consumer = module.cv1.conv, module.cv2.conv
        bn = getattr(module.cv1, "bn", None)
        if not isinstance(bn, torch.nn.BatchNorm2d) or producer.groups != 1 or consumer.groups != 1:
            raise ValueError(f"Bottleneck {name} tidak mendukung kebijakan hidden-channel konservatif.")
        if producer.out_channels != consumer.in_channels or producer.out_channels != bn.num_features:
            raise ValueError(f"Dimensi hidden channel {name} tidak konsisten.")
        allowed[name + ".cv1.conv"] = {"root": producer, "bn": bn, "consumer": consumer,
                                        "initial_channels": producer.out_channels, "block": name}
    if not allowed:
        raise ValueError("Tidak ditemukan hidden-channel group yang eligible.")
    backbone_count = len(cfg["backbone"])
    stages = []
    for i, module in enumerate(core.model):
        stages.append({"index": i, "type": type(module).__name__, "from": module.f,
                       "region": "detection_head" if module is head else ("backbone" if i < backbone_count else "neck"),
                       "parameters": sum(p.numel() for p in module.parameters())})
    inventory = [{"name": name, "type": type(module).__name__,
                  "residual_add": bool(getattr(module, "add", False))}
                 for name, module in modules.items()
                 if isinstance(module, (torch.nn.Conv2d, torch.nn.BatchNorm2d, Bottleneck, C2PSA, Attention, AAttn))
                 or type(module).__name__ in {"Concat", "C3k2", "SPPF"}]
    protection = []
    for name, module in modules.items():
        if isinstance(module, torch.nn.Conv2d) and name not in allowed:
            reason = "head one2many/one2one utuh" if name.startswith(head_name + ".") else "dimensi CSP/residual/attention/antartahap dilindungi"
            protection.append({"name": name, "reason": reason,
                               "protected_dimension": "all" if name.startswith(head_name + ".") else "out_channels",
                               "dependent_input_pruning_allowed": any(module is entry["consumer"] for entry in allowed.values())})
    return {"head": head, "head_name": head_name, "eligible": allowed, "modules": modules,
            "report": {"stages": stages, "module_inventory": inventory, "protected_layers": protection,
                       "eligible_layers": list(allowed), "head_feature_sources": list(head.f),
                       "class_names": names, "nc": head.nc, "reg_max": head.reg_max, "nl": head.nl,
                       "end2end": head.end2end, "unknown_policy": "fail explicitly"}}


def raw_branches(output):
    """Ambil raw kedua cabang tanpa mengganti forward atau algoritme deteksi."""
    if not isinstance(output, tuple) or len(output) != 2 or not isinstance(output[1], dict):
        raise ValueError("Output eval YOLO26 harus (decoded, raw_predictions).")
    branches = output[1]
    if set(branches) != {"one2many", "one2one"}:
        raise ValueError(f"Cabang YOLO26 tidak lengkap: {list(branches)}")
    return branches


def trace_outputs(output):
    """Root autograd mencakup boxes/scores kedua cabang, tanpa top-k/NMS."""
    branches = raw_branches(output)
    return [branches[branch][key] for branch in ("one2many", "one2one") for key in ("boxes", "scores")]


def finite_tensors(value):
    """Validasi semua tensor, termasuk feature maps dalam output nested."""
    if isinstance(value, torch.Tensor):
        if not torch.isfinite(value).all():
            raise ValueError("Output model mengandung NaN/Inf.")
    elif isinstance(value, dict):
        for item in value.values():
            finite_tensors(item)
    elif isinstance(value, (tuple, list)):
        for item in value:
            finite_tensors(item)


def forward_snapshot(core, image):
    """Forward FP32, cek kontrak raw/decoded, simpan tensor kecil untuk restore parity."""
    core.eval()
    with torch.no_grad():
        output = core(image)
    finite_tensors(output)
    branches = raw_branches(output)
    head = core.model[-1]
    snapshot = {"decoded": output[0].detach().cpu().clone()}
    anchors = None
    for branch in ("one2many", "one2one"):
        boxes, scores = branches[branch]["boxes"], branches[branch]["scores"]
        if boxes.ndim != 3 or scores.ndim != 3 or boxes.shape[:2] != (image.shape[0], 4 * head.reg_max):
            raise ValueError(f"Bentuk boxes {branch} invalid: {tuple(boxes.shape)}")
        if scores.shape[:2] != (image.shape[0], head.nc) or boxes.shape[2] != scores.shape[2]:
            raise ValueError(f"Bentuk scores {branch} tidak cocok dengan jumlah kelas.")
        if anchors is not None and anchors != boxes.shape[2]:
            raise ValueError("Jumlah anchor kedua cabang berbeda.")
        anchors = boxes.shape[2]
        snapshot[branch + "/boxes"] = boxes.detach().cpu().clone()
        snapshot[branch + "/scores"] = scores.detach().cpu().clone()
    decoded = snapshot["decoded"]
    if head.end2end:
        if decoded.ndim != 3 or decoded.shape[0] != image.shape[0] or decoded.shape[-1] != 6:
            raise ValueError("Output end-to-end bukan [batch, detections, 6].")
        ids = decoded[..., 5]
        if ((ids < 0) | (ids >= head.nc) | (ids != ids.floor())).any():
            raise ValueError("Indeks kelas pada output deteksi invalid.")
    elif decoded.ndim != 3 or decoded.shape[:2] != (image.shape[0], head.nc + 4):
        raise ValueError("Output one-to-many tidak memiliki dimensi nc+4.")
    return snapshot


def model_measurements(core, imgsz):
    """Hitung Conv/BN dan attention matmul tanpa menghilangkan child Conv attention."""
    parameters = sum(p.numel() for p in core.parameters())
    copy = deepcopy(core).eval()
    parameter = next(copy.parameters())
    sample = torch.zeros(1, 3, imgsz, imgsz, device=parameter.device, dtype=torch.float32)
    attention_macs = []
    def count_attention(module, inputs, output):
        counter = SimpleNamespace(total_ops=0, num_heads=module.num_heads, head_dim=module.head_dim,
                                  key_dim=getattr(module, "key_dim", module.head_dim), area=getattr(module, "area", 1))
        _attention_ops(counter, inputs, output)
        attention_macs.append(float(counter.total_ops))
    handles = [module.register_forward_hook(count_attention) for module in copy.modules()
               if isinstance(module, (Attention, AAttn))]
    try:
        with torch.no_grad():
            # A custom Attention handler in classic THOP would hide its child Conv/BN counts.
            macs, _ = thop_profile(copy, inputs=(sample,), verbose=False)
    finally:
        for handle in handles:
            handle.remove()
    gflops = (float(macs) + sum(attention_macs)) * 2 / 1e9
    if not math.isfinite(gflops) or gflops <= 0:
        raise RuntimeError("Profiling GFLOPs gagal/bernilai nol; cek thop dan source dependency.")
    return {"parameters": parameters, "gflops": gflops, "precision": "FP32", "fusion": False}


def head_snapshot(head):
    """Salin parameter/buffer serta kontrak dimensi seluruh head yang dilindungi."""
    return {"state": {key: value.detach().cpu().clone() for key, value in head.state_dict().items()},
            "contract": (head.nc, head.no, head.nl, head.reg_max, head.end2end)}


def assert_head_unchanged(head, before):
    """Pruning tersembunyi tidak boleh menyentuh bobot atau dimensi head."""
    after = head_snapshot(head)
    if after["contract"] != before["contract"] or after["state"].keys() != before["state"].keys():
        raise ValueError("Kontrak/struktur detection head berubah.")
    if any(not torch.equal(after["state"][key], value) for key, value in before["state"].items()):
        raise ValueError("Parameter/buffer detection head berubah.")

In [ ]:
def assert_layer_consistency(core):
    """Dimensi atribut modul harus sesuai dengan tensor parameter/buffer aktual."""
    for name, module in core.named_modules():
        if isinstance(module, torch.nn.Conv2d):
            expected = (module.out_channels, module.in_channels // module.groups, *module.kernel_size)
            if tuple(module.weight.shape) != expected or module.in_channels % module.groups or module.out_channels % module.groups:
                raise ValueError(f"Bentuk Conv invalid pada {name}.")
        elif isinstance(module, torch.nn.BatchNorm2d):
            if any(t.numel() != module.num_features for t in (module.weight, module.bias, module.running_mean, module.running_var)):
                raise ValueError(f"Bentuk BatchNorm invalid pada {name}.")

## 06 — Audit Dependency Graph dan Eligible Layer
Conservative mempertahankan hidden Bottleneck V1. Extended memeriksa seluruh group graph
dan hanya menerima operasi yang didukung, tanpa indeks layer hardcoded. Head dan envelope
attention utuh; split/reshape dan mapping tidak valid ditolak. Group residual yang mengikat
beberapa output dihitung sekali pada denominator channel independen, bukan setiap Conv.
Daftar lengkap accepted/rejected groups dan alasan ditulis sebelum channel dihapus.

In [ ]:
"""Standalone LAMP V2 policy source, embedded verbatim in the Kaggle notebook.

This file is an authoring source, not a notebook runtime dependency.
"""
import math
import warnings

import torch
import torch_pruning as tp
from ultralytics.nn.modules import Bottleneck, C2f, C3k2, Detect
from ultralytics.nn.modules.block import C2PSA, Attention, PSABlock
from ultralytics.nn.tasks import DetectionModel


def _v2_trace_outputs(output):
    if not isinstance(output, tuple) or len(output) != 2 or not isinstance(output[1], dict):
        raise ValueError("Output YOLO26 eval harus (decoded, raw).")
    if set(output[1]) != {"one2many", "one2one"}:
        raise ValueError("Graph wajib mencakup raw output kedua cabang.")
    return [output[1][branch][key] for branch in ("one2many", "one2one")
            for key in ("boxes", "scores")]


def _v2_policy_structure(core, model_variant=None):
    if not isinstance(core, DetectionModel):
        raise TypeError("Policy memerlukan DetectionModel YOLO26 unfused.")
    modules = dict(core.named_modules())
    heads = [(name, module) for name, module in modules.items() if isinstance(module, Detect)]
    if len(heads) != 1:
        raise ValueError("Diperlukan satu Detect head.")
    head_name, head = heads[0]
    if any(getattr(head, name, None) is None for name in ("cv2", "cv3", "one2one_cv2", "one2one_cv3")):
        raise ValueError("Detection head kedua cabang harus unfused dan lengkap.")
    first = next(module for module in modules.values() if isinstance(module, torch.nn.Conv2d))
    actual_variant = core.yaml.get("scale")
    if actual_variant in (None, ""):
        actual_variant = {16: "n", 32: "s"}.get(first.out_channels)
    if model_variant is not None and model_variant not in ("n", "s"):
        raise ValueError("Model variant harus n atau s.")
    if ("yolo26" not in str(core.yaml.get("yaml_file", "")).lower()
            or core.yaml.get("reg_max") != 1 or actual_variant not in ("n", "s")
            or first.out_channels != {"n": 16, "s": 32}.get(actual_variant)):
        raise ValueError("V2 memerlukan arsitektur standar YOLO26n/S dengan stem 16/32 yang sesuai.")
    if model_variant is not None and model_variant != actual_variant:
        raise ValueError(f"Model variant tidak cocok: diminta {model_variant}, checkpoint {actual_variant}.")
    envelopes = [(name, module) for name, module in modules.items()
                 if isinstance(module, (Detect, C2PSA, PSABlock, Attention))
                 or type(module).__name__ == "AAttn"]
    protected = {}
    for name, module in modules.items():
        for prefix, envelope in envelopes:
            if name == prefix or name.startswith(prefix + "."):
                protected[module] = "detection_head_all_dimensions" if isinstance(envelope, Detect) else "attention_envelope_all_dimensions"
                break
    split_outputs = {}
    for name, module in modules.items():
        if isinstance(module, (C2f, C3k2)):
            for child in (module.cv1.conv, getattr(module.cv1, "bn", None)):
                if child is not None:
                    split_outputs[child] = "CSP_split_output_dimension"
    conservative = {}
    for name, module in modules.items():
        if not isinstance(module, Bottleneck) or module in protected:
            continue
        producer, consumer = module.cv1.conv, module.cv2.conv
        bn = getattr(module.cv1, "bn", None)
        if (not isinstance(bn, torch.nn.BatchNorm2d) or producer.groups != 1 or consumer.groups != 1
                or producer.out_channels != consumer.in_channels or producer.out_channels != bn.num_features):
            raise ValueError(f"Bottleneck hidden channel tidak didukung: {name}")
        conservative[producer] = {"name": name + ".cv1.conv", "bn": bn, "consumer": consumer}
    if not conservative:
        raise ValueError("Tidak ada kandidat konservatif.")
    return {"model_variant": actual_variant, "modules": modules, "names": {module: name for name, module in modules.items()},
            "head": head, "head_name": head_name, "protected": protected,
            "split_outputs": split_outputs, "conservative": conservative}


def protected_policy_snapshot(core, model_variant=None):
    """Snapshot seluruh bobot, buffer, dan kontrak head/attention yang dilindungi."""
    structure = _v2_policy_structure(core, model_variant)
    result = {}
    for name, module in structure["modules"].items():
        if isinstance(module, (Detect, C2PSA, PSABlock, Attention)) or type(module).__name__ == "AAttn":
            result[name] = {"state": {key: value.detach().cpu().clone() for key, value in module.state_dict().items()},
                            "dimensions": {key: getattr(module, key) for key in ("nc", "no", "nl", "reg_max", "end2end", "c")
                                           if hasattr(module, key)}}
    return result


def assert_policy_protected_unchanged(core, snapshot):
    modules = dict(core.named_modules())
    for name, entry in snapshot.items():
        if name not in modules:
            raise ValueError(f"Envelope terlindungi hilang: {name}")
        module = modules[name]
        after = module.state_dict()
        if after.keys() != entry["state"].keys() or any(
                not torch.equal(value.detach().cpu(), entry["state"][key]) for key, value in after.items()):
            raise ValueError(f"Bobot/buffer envelope terlindungi berubah: {name}")
        if any(getattr(module, key) != value for key, value in entry["dimensions"].items()):
            raise ValueError(f"Kontrak envelope terlindungi berubah: {name}")


def _v2_output_signature(group, graph, names):
    """Identitas satu group adalah seluruh dimensi Conv-output yang terikat."""
    return tuple(sorted(names[dep.target.module] for dep, _ in group
                        if isinstance(dep.target.module, torch.nn.Conv2d)
                        and graph.is_out_channel_pruning_fn(dep.handler)))


def _v2_validate_group(group, graph, structure, policy):
    root = group[0].dep.target.module
    root_name = structure["names"].get(root, "<unknown>")
    entry = structure["conservative"].get(root)
    if policy == "conservative" and entry is None:
        raise ValueError("root_not_conservative_hidden_channel")
    for dep, indices in group:
        module, handler = dep.target.module, dep.handler
        name = structure["names"].get(module, type(module).__name__)
        output = graph.is_out_channel_pruning_fn(handler)
        if module in structure["protected"]:
            raise ValueError(f"{name}: {structure['protected'][module]}")
        if output and module in structure["split_outputs"]:
            raise ValueError(f"{name}: {structure['split_outputs'][module]}")
        if isinstance(module, torch.nn.Conv2d):
            if not output and not graph.is_in_channel_pruning_fn(handler):
                raise ValueError(f"{name}: unknown_conv_pruning_handler")
            if module.groups != 1:
                raise ValueError(f"{name}: grouped_or_depthwise_conv_unsupported")
            if policy == "conservative" and not (
                    (module is root and output) or (module is entry["consumer"] and graph.is_in_channel_pruning_fn(handler))):
                raise ValueError(f"{name}: outside_conservative_hidden_channel")
        elif isinstance(module, torch.nn.BatchNorm2d):
            if policy == "conservative" and module is not entry["bn"]:
                raise ValueError(f"{name}: outside_conservative_batchnorm")
        else:
            kind = type(module).__name__
            if kind not in {"_ElementWiseOp", "_ConcatOp"}:
                raise ValueError(f"{name}: unsupported_operation_{kind}")
            if kind == "_ElementWiseOp":
                operation = str(module)
                supported = ("SiluBackward", "AddBackward", "MaxPool2DWithIndicesBackward", "UpsampleNearest2DBackward", "CloneBackward")
                if not any(token.lower() in operation.lower() for token in supported):
                    raise ValueError(f"{name}: unsupported_elementwise_{operation}")
        # Torch-Pruning maps concat offsets and coupled residual identities. Reject invalid maps.
        mapped = [int(index) for index in indices]
        if len(mapped) != len(set(mapped)) or any(index < 0 for index in mapped):
            raise ValueError(f"{name}: invalid_or_duplicate_index_mapping")
        channels = graph.get_out_channels(module) if output else graph.get_in_channels(module)
        if channels is not None and any(index >= channels for index in mapped):
            raise ValueError(f"{name}: dependency_index_out_of_bounds")
    return root_name


def _v2_channel_ledger(structure):
    """Track each physical model dimension independently in original index space."""
    ledger = {}
    for name, module in structure["modules"].items():
        if isinstance(module, torch.nn.Conv2d):
            dimensions = {"in": module.in_channels, "out": module.out_channels}
        elif isinstance(module, torch.nn.BatchNorm2d):
            dimensions = {"channel": module.num_features}
        else:
            continue
        for dimension, count in dimensions.items():
            ledger[(module, dimension)] = {
                "layer": name, "module_type": type(module).__name__, "dimension": dimension,
                "channels_before": count, "kept": list(range(count)), "group_ids": set()}
    return ledger


def _v2_dependency_dimension(dependency, graph):
    module = dependency.target.module
    if isinstance(module, torch.nn.BatchNorm2d):
        return module, "channel"
    if isinstance(module, torch.nn.Conv2d):
        return module, "out" if graph.is_out_channel_pruning_fn(dependency.handler) else "in"
    return None


def _v2_refresh_channel_report(audit):
    mappings, masks = [], []
    for entry in audit["channel_ledger"].values():
        kept = set(entry["kept"])
        removed = [index for index in range(entry["channels_before"]) if index not in kept]
        row = {key: entry[key] for key in ("layer", "module_type", "dimension", "channels_before")}
        row.update(channels_after=len(entry["kept"]), kept_original_indices=list(entry["kept"]),
                   removed_original_indices=removed, dependency_group_ids=sorted(entry["group_ids"]))
        mappings.append(row)
        masks.append({"layer": row["layer"], "module_type": row["module_type"],
                      "dimension": row["dimension"], "channels_before": row["channels_before"],
                      "channels_after": row["channels_after"],
                      "dependency_group_ids": row["dependency_group_ids"],
                      "kept_mask": [index in kept for index in range(entry["channels_before"])]})
    audit["report"]["channel_mapping"] = mappings
    audit["report"]["channel_masks"] = masks


def _v2_capacity(entry):
    current = entry["root"].out_channels
    capacity = current - 1
    for module, initial in entry["output_dimensions"].items():
        capacity = min(capacity, math.floor(initial * 0.5) - (initial - module.out_channels))
    return max(0, capacity)


def build_policy_pruner(core, image, policy, ratio, model_variant="n"):
    """Discover, audit and canonicalize dependencies before global LAMP ranking."""
    if policy not in ("conservative", "extended"):
        raise ValueError("Policy harus conservative atau extended.")
    if not 0 < float(ratio) <= 0.5:
        raise ValueError("Rasio harus di dalam (0, 0.50].")
    core.eval().requires_grad_(True)
    structure = _v2_policy_structure(core, model_variant)
    importance = tp.importance.LAMPImportance(p=2, group_reduction="mean", bias=False)
    initial_ignored = [module for module in structure["modules"].values()
                       if isinstance(module, torch.nn.Conv2d) and
                       (module in structure["protected"] or module in structure["split_outputs"]
                        or (policy == "conservative" and module not in structure["conservative"]))]
    with torch.enable_grad(), warnings.catch_warnings(record=True) as captured:
        warnings.simplefilter("always")
        pruner = tp.pruner.BasePruner(core, image, importance=importance,
            global_pruning=True, pruning_ratio=float(ratio), iterative_steps=1,
            max_pruning_ratio=0.50, round_to=None, root_module_types=[torch.nn.Conv2d],
            ignored_layers=initial_ignored,
            forward_fn=lambda model, example: model(example), output_transform=_v2_trace_outputs)
    graph = pruner.DG
    eligible, rejected, seen, seen_outputs = {}, [], set(), set()
    # Inspect every Conv root explicitly, including rejected roots. The later canonical
    # scan follows BasePruner's own graph order to preserve V1 ranking semantics.
    for name, module in structure["modules"].items():
        if not isinstance(module, torch.nn.Conv2d):
            continue
        if module in initial_ignored:
            reason = structure["protected"].get(module) or structure["split_outputs"].get(module) or "root_not_conservative_hidden_channel"
            rejected.append({"root": name, "reason": reason})
            continue
        group = graph.get_pruning_group(module, tp.prune_conv_out_channels, list(range(module.out_channels)))
        try:
            _v2_validate_group(group, graph, structure, policy)
        except ValueError as error:
            rejected.append({"root": name, "reason": str(error)})
    rejected_roots = {entry["root"] for entry in rejected}
    ignored = [module for name, module in structure["modules"].items()
               if isinstance(module, torch.nn.Conv2d) and name in rejected_roots]
    for group in graph.get_all_groups(ignored_layers=ignored, root_module_types=[torch.nn.Conv2d]):
        root_name = _v2_validate_group(group, graph, structure, policy)
        signature = _v2_output_signature(group, graph, structure["names"])
        if not signature or signature in seen:
            continue
        overlap = seen_outputs.intersection(signature)
        if overlap:
            raise ValueError(f"Canonical dependency groups overlap output dimensions: {sorted(overlap)}")
        seen.add(signature)
        seen_outputs.update(signature)
        root = group[0].dep.target.module
        score = importance(group)
        if score is None or score.numel() != root.out_channels or not torch.isfinite(score).all():
            raise ValueError(f"Importance LAMP invalid: {root_name}")
        dimensions = {dep.target.module: int(dep.target.module.out_channels)
                      for dep, _ in group if isinstance(dep.target.module, torch.nn.Conv2d)
                      and graph.is_out_channel_pruning_fn(dep.handler)}
        membership = {dep.target.module: ("out" if graph.is_out_channel_pruning_fn(dep.handler) else "in")
                      for dep, _ in group
                      if isinstance(dep.target.module, (torch.nn.Conv2d, torch.nn.BatchNorm2d))}
        capacity = min(root.out_channels - 1, *(math.floor(initial * 0.5) for initial in dimensions.values()))
        permitted = set(torch.argsort(score, stable=True)[:capacity].detach().cpu().tolist())
        eligible[root] = {"name": root_name, "root": root, "signature": signature,
                          "initial_channels": root.out_channels, "output_dimensions": dimensions,
                          "member_dimensions": membership,
                          "initial_lamp_scores": score.detach().cpu().tolist(),
                          "initial_capacity_mask": [index in permitted for index in range(root.out_channels)],
                          "group_id": f"{policy}-group-{len(eligible):03d}",
                          "dependencies": [str(dep) for dep, _ in group]}
    if not eligible:
        raise ValueError("Audit tidak menemukan group aman untuk policy ini.")
    pruner.ignored_layers = ignored
    audit = {"structure": structure, "eligible": eligible, "policy": policy, "ratio": float(ratio),
             "model_variant": structure["model_variant"], "channel_ledger": _v2_channel_ledger(structure),
             "independent_channels_before": sum(entry["initial_channels"] for entry in eligible.values()),
             "physical_conv_output_channels_before": sum(sum(entry["output_dimensions"].values()) for entry in eligible.values())}
    audit["report"] = {"model_variant": structure["model_variant"], "policy": policy,
         "requested_ratio": float(ratio), "max_pruning_ratio": 0.5,
         "importance": "LAMPImportance(p=2, group_reduction='mean', bias=False)",
         "ranking": "BasePruner global one-step; capacity exclusions are +inf sentinels, finite eligible LAMP scores unchanged",
         "group_count": len(eligible), "independent_channels_before": audit["independent_channels_before"],
         "physical_conv_output_channels_before": audit["physical_conv_output_channels_before"],
         "eligible_groups": [{key: value for key, value in entry.items() if key not in ("root", "output_dimensions", "member_dimensions")}
                             for entry in eligible.values()],
         "rejected_groups": rejected, "warnings": [str(item.message) for item in captured],
         "lamp_scores": [{"group_id": entry["group_id"], "root": entry["name"],
                          "channel_idx": index, "score": score,
                          "capacity_eligible": entry["initial_capacity_mask"][index], "removed": False}
                         for entry in eligible.values() for index, score in enumerate(entry["initial_lamp_scores"])],
         "channel_mapping_status": "RECORDED_FROM_LIVE_DEPENDENCY_GRAPH",
         "channel_mapping_index_space": "Original model dimension indices; current indices translated before every successful prune"}
    for entry in eligible.values():
        group = graph.get_pruning_group(entry["root"], tp.prune_conv_out_channels,
                                        list(range(entry["initial_channels"])))
        for dependency, _ in group:
            dimension = _v2_dependency_dimension(dependency, graph)
            if dimension is not None:
                audit["channel_ledger"][dimension]["group_ids"].add(entry["group_id"])
    _v2_refresh_channel_report(audit)

    def capacity_limited_lamp(group):
        _v2_validate_group(group, graph, structure, policy)
        entry = eligible.get(group[0].dep.target.module)
        if entry is None:
            raise ValueError("Proposal berakar di luar group yang diaudit.")
        score = importance(group)
        current = entry["root"].out_channels
        if score is None or score.numel() != current or not torch.isfinite(score).all():
            raise ValueError("Importance LAMP tidak finite sebelum cap.")
        # Bound every coupled output dimension, preserving the original LAMP score.
        capacity = _v2_capacity(entry)
        permitted = torch.argsort(score, stable=True)[:capacity]
        capped = torch.full_like(score, float("inf"))
        capped[permitted] = score[permitted]
        return capped
    pruner.importance = capacity_limited_lamp
    return pruner, audit


def apply_policy_pruning(pruner, audit):
    """Audit each concrete proposal before irreversible tensor shape mutation."""
    history = []
    graph, structure = pruner.DG, audit["structure"]
    # The runtime report retains this live list even if a later proposal raises.
    audit["report"]["pruning_history"] = history
    audit["report"]["pruning_status"] = "IN_PROGRESS"
    for group in pruner.step(interactive=True):
        audit["report"]["last_proposal"] = {
            "root": structure["names"].get(group[0].dep.target.module, "<unknown>"),
            "removed_indices": [int(index) for index in group[0].idxs]}
        name = _v2_validate_group(group, graph, structure, audit["policy"])
        entry = audit["eligible"].get(group[0].dep.target.module)
        if entry is None:
            raise ValueError("Group proposal tidak terdapat pada audit.")
        indices = [int(index) for index in group[0].idxs]
        if not indices:
            continue
        if _v2_output_signature(group, graph, structure["names"]) != entry["signature"]:
            raise ValueError("Dimensi output proposal berbeda dari group canonical yang diaudit.")
        if not graph.check_pruning_group(group):
            raise ValueError(f"Dependency group tidak valid: {name}")
        mapped_dimensions = {}
        for dependency, mapped in group:
            dimension = _v2_dependency_dimension(dependency, graph)
            if dimension is None:
                continue
            # A residual/concat may reach a physical dimension along multiple paths.
            # The ledger records its union once, in the current dimension's space.
            mapped_dimensions.setdefault(dimension, set()).update(int(index) for index in mapped)
        dimension_records = []
        for dimension, mapped in mapped_dimensions.items():
            ledger = audit["channel_ledger"][dimension]
            current_indices = sorted(mapped)
            if any(index < 0 or index >= len(ledger["kept"]) for index in current_indices):
                raise ValueError(f"Ledger dependency index out of bounds: {ledger['layer']}:{dimension[1]}")
            dimension_records.append({"layer": ledger["layer"], "module_type": ledger["module_type"],
                "dimension": dimension[1], "current_indices": current_indices,
                "original_indices": [ledger["kept"][index] for index in current_indices],
                "channels_before": len(ledger["kept"]), "channels_after": len(ledger["kept"]) - len(mapped)})
        physical_removed = 0
        touched = []
        for (module, dimension), mapped in mapped_dimensions.items():
            if isinstance(module, torch.nn.Conv2d) and dimension == "out":
                if module not in entry["output_dimensions"]:
                    raise ValueError("Proposal mengubah output di luar group canonical.")
                initial = entry["output_dimensions"][module]
                remaining = module.out_channels - len(mapped)
                if remaining < 1 or initial - remaining > math.floor(initial * 0.5):
                    raise ValueError(f"Cap 50%/minimum satu channel dilanggar: {structure['names'][module]}")
                physical_removed += len(mapped)
                touched.append({"layer": structure["names"][module], "before": module.out_channels, "after": remaining})
        before = entry["root"].out_channels
        root_ledger = audit["channel_ledger"][(entry["root"], "out")]
        original_root_indices = [root_ledger["kept"][index] for index in indices]
        record = {"root": name, "group_id": entry["group_id"], "removed_indices": indices,
                  "removed_original_indices": original_root_indices, "mapped_dimensions": dimension_records,
                  "channels_before": before, "channels_after": before - len(indices),
                  "physical_conv_output_channels_removed": physical_removed,
                  "coupled_conv_output_count": len(touched),
                  "output_changes": touched, "dependencies": [str(dep) for dep, _ in group]}
        group.prune()
        for (module, dimension), mapped in mapped_dimensions.items():
            ledger = audit["channel_ledger"][(module, dimension)]
            ledger["kept"] = [original for index, original in enumerate(ledger["kept"]) if index not in mapped]
            ledger["group_ids"].add(entry["group_id"])
            actual = (module.in_channels if dimension == "in" else module.out_channels) if isinstance(
                module, torch.nn.Conv2d) else module.num_features
            if len(ledger["kept"]) != actual:
                raise ValueError(f"Ledger/model channel mismatch: {ledger['layer']}:{dimension}")
        removed_set = set(original_root_indices)
        for row in audit["report"]["lamp_scores"]:
            if row["group_id"] == entry["group_id"] and row["channel_idx"] in removed_set:
                row["removed"] = True
        _v2_refresh_channel_report(audit)
        history.append(record)
        audit["report"]["independent_channels_after"] = sum(entry["root"].out_channels for entry in audit["eligible"].values())
        audit["report"]["independent_channels_removed"] = sum(item["channels_before"] - item["channels_after"] for item in history)
        audit["report"]["physical_conv_output_channels_removed"] = sum(item["physical_conv_output_channels_removed"] for item in history)
    if not history:
        raise RuntimeError("Tidak ada channel struktural dihapus.")
    audit["report"]["pruning_status"] = "PASS"
    return history


def measure_layer_inventory(core, image, audit, policy):
    """Per-layer Conv/BN estimates at the actual input size, on an unfused model."""
    shapes, handles = {}, []
    for name, module in core.named_modules():
        if isinstance(module, (torch.nn.Conv2d, torch.nn.BatchNorm2d)):
            def hook(layer, args, output, name=name):
                shapes[name] = list(output.shape)
            handles.append(module.register_forward_hook(hook))
    try:
        with torch.no_grad():
            core(image)
    finally:
        for handle in handles:
            handle.remove()
    structure = audit["structure"]
    membership, member_dimensions = {}, {}
    for entry in audit["eligible"].values():
        for module, dimension in entry["member_dimensions"].items():
            membership.setdefault(module, []).append(entry["group_id"])
            member_dimensions.setdefault(module, set()).add(dimension)
    rejected = {item["root"]: item["reason"] for item in audit["report"]["rejected_groups"]}
    backbone_count = len(core.yaml.get("backbone", []))
    rows = []
    for name, module in core.named_modules():
        if not isinstance(module, (torch.nn.Conv2d, torch.nn.BatchNorm2d)):
            continue
        shape = shapes.get(name, [])
        if isinstance(module, torch.nn.Conv2d):
            inputs, outputs = module.in_channels, module.out_channels
            operations = (2 * math.prod(shape) * (inputs // module.groups) * math.prod(module.kernel_size)) if shape else None
        else:
            inputs = outputs = module.num_features
            operations = 2 * math.prod(shape) if shape else None
        stage = next((i for i, block in enumerate(core.model)
                      if any(child is module for child in block.modules())), None)
        reason = structure["protected"].get(module) or structure["split_outputs"].get(module) or rejected.get(name)
        rows.append({"layer_name": name, "module_type": type(module).__name__,
                     "region": "detection_head" if module in structure["protected"] and str(reason).startswith("detection")
                               else ("backbone" if stage is not None and stage < backbone_count else "neck"),
                     "in_channels": inputs, "out_channels": outputs,
                     "parameters": sum(parameter.numel() for parameter in module.parameters(recurse=False)),
                     "estimated_flops": operations, "flops_estimate_method": "Conv:2*elements*Cin/groups*Kh*Kw; BN:2*elements; functional attention excluded",
                     "policy": policy, "eligible": module in membership,
                     "protection_reason": reason or ("not_in_eligible_dependency_group" if module not in membership else ""),
                     "dependency_group": ";".join(membership.get(module, [])),
                     "eligible_dimensions": ";".join(sorted(member_dimensions.get(module, set()))), "output_shape": shape})
    return rows

In [ ]:
"""Standalone analysis helpers embedded in notebook 03 by its generator.

All figures describe measured structural changes, not activation feature maps.
Missing historical mappings, scores, and validation values are never fabricated.
"""
import csv as _eda_csv
import json as _eda_json
import math as _eda_math
from pathlib import Path as _EdaPath

import matplotlib.pyplot as _eda_plt
import numpy as _eda_np
from matplotlib.colors import ListedColormap as _EdaColors


def _eda_write_json(path, value):
    path = _EdaPath(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(_eda_json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")


def _eda_write_csv(path, rows):
    path = _EdaPath(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    keys = list(dict.fromkeys(key for row in rows for key in row))
    with path.open("w", newline="", encoding="utf-8") as stream:
        writer = _eda_csv.DictWriter(stream, fieldnames=keys)
        writer.writeheader()
        writer.writerows({key: _eda_json.dumps(value, ensure_ascii=False) if isinstance(value, (dict, list)) else value
                         for key, value in row.items()} for row in rows)


def _eda_finite(value):
    return isinstance(value, (int, float)) and not isinstance(value, bool) and _eda_math.isfinite(value)


def _eda_save_figure(fig, path):
    path = _EdaPath(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path.with_suffix(".png"), dpi=int(globals().get("EDA_DPI", 300)), bbox_inches="tight")
    fig.savefig(path.with_suffix(".svg"), bbox_inches="tight")
    _eda_plt.close(fig)
    return {"png": path.with_suffix(".png").name, "svg": path.with_suffix(".svg").name}


def layer_eda_rows(inventory_before, inventory_after):
    """Join actual unfused inventory by module name, preserving baseline order."""
    previous = {row["layer_name"]: row for row in inventory_before}
    current = {row["layer_name"]: row for row in inventory_after}
    if len(previous) != len(inventory_before) or len(current) != len(inventory_after):
        raise ValueError("EDA inventory has duplicate module names.")
    if set(previous) != set(current):
        raise ValueError("EDA before/after module names differ; layer alignment is unsafe.")
    rows = []
    for name, before in previous.items():
        after = current[name]
        if before["module_type"] != after["module_type"]:
            raise ValueError(f"EDA module type changed at {name}.")
        if after["out_channels"] > before["out_channels"] or after["in_channels"] > before["in_channels"]:
            raise ValueError(f"Pruning unexpectedly increased channels at {name}.")
        parts = name.split(".")
        stage = before.get("stage", ".".join(parts[:2]))
        row = {"layer_name": name, "module_type": before["module_type"], "stage": stage,
               "region": before.get("region", "unknown"), "policy": before.get("policy"),
               "eligible": before.get("eligible", False), "protection_reason": before.get("protection_reason", ""),
               "dependency_group": before.get("dependency_group", "")}
        for key in ("in_channels", "out_channels", "parameters", "estimated_flops"):
            row[key + "_before"] = before.get(key)
            row[key + "_after"] = after.get(key)
            base, value = before.get(key), after.get(key)
            row[key + "_removed"] = base - value if _eda_finite(base) and _eda_finite(value) else None
            row[key + "_reduction_pct"] = 100 * (base - value) / base if _eda_finite(base) and _eda_finite(value) and base else None
        if (row["protection_reason"] and str(row["protection_reason"]).startswith(("detection", "attention"))
                and (row["out_channels_removed"] or row["in_channels_removed"])):
            raise ValueError(f"Protected head/attention channel dimension changed at {name}.")
        rows.append(row)
    return rows


def stage_eda_rows(rows):
    """Physical Conv-output sums; these are distinct from independent group channels."""
    stages = {}
    for row in rows:
        key = (str(row["stage"]), row["region"])
        stage = stages.setdefault(key, {"stage": row["stage"], "region": row["region"], "conv_layers": 0,
                                      "physical_conv_outputs_before": 0, "physical_conv_outputs_after": 0,
                                      "parameters_before": 0, "parameters_after": 0,
                                      "estimated_flops_before": 0, "estimated_flops_after": 0})
        if row["module_type"] == "Conv2d":
            stage["conv_layers"] += 1
            for suffix in ("before", "after"):
                stage["physical_conv_outputs_" + suffix] += row["out_channels_" + suffix]
        for metric in ("parameters", "estimated_flops"):
            for suffix in ("before", "after"):
                value = row[metric + "_" + suffix]
                if _eda_finite(value):
                    stage[metric + "_" + suffix] += value
    for row in stages.values():
        row["physical_conv_outputs_removed"] = row["physical_conv_outputs_before"] - row["physical_conv_outputs_after"]
        for key in ("physical_conv_outputs", "parameters", "estimated_flops"):
            base, after = row[key + "_before"], row[key + "_after"]
            row[key + "_reduction_pct"] = 100 * (base - after) / base if base else None
    return list(stages.values())


def make_channel_figure(rows, title="Channel comparison"):
    """Three panels share exact Conv order and Y limits, as in the reference image."""
    if not rows or any(row["module_type"] != "Conv2d" for row in rows):
        raise ValueError("Channel diagram requires a nonempty ordered Conv2d inventory.")
    count = len(rows)
    width = max(16, min(32, count * 0.16))
    fig, axes = _eda_plt.subplots(3, 1, figsize=(width, 12), sharex=True, sharey=True)
    indices = _eda_np.arange(count)
    before = [row["out_channels_before"] for row in rows]
    after = [row["out_channels_after"] for row in rows]
    upper = max(max(before), max(after), 1) * 1.12
    axes[0].bar(indices, before, color="blue", width=0.82, label="Before Pruning")
    axes[1].bar(indices, after, color="red", width=0.82, label="After Pruning")
    axes[2].bar(indices, before, color="blue", width=0.82, label="Before Pruning")
    axes[2].bar(indices, after, color="red", width=0.82, label="After Pruning")
    for ax, heading in zip(axes, ("Before Pruning", "After Pruning", "Channel Comparison")):
        ax.set_title(heading)
        ax.set_ylabel("Output channels")
        ax.set_ylim(0, upper)
        ax.set_xlim(-0.8, count - 0.2)
        ax.grid(axis="y", alpha=0.2)
        ax.set_axisbelow(True)
        ax.legend(loc="upper right")
        ax.tick_params(axis="x", labelbottom=True)
        ax.set_xticks(indices, [row["layer_name"] for row in rows], rotation=90, fontsize=6 if count > 32 else 8)
        ax.set_xlabel("Conv module (baseline order)")
    fig.suptitle(title, y=1.005, fontsize=14)
    fig.tight_layout()
    return fig


def _eda_channel_plots(conv_rows, folder, title):
    _eda_write_csv(folder / "channel_plot_data.csv", conv_rows)
    files = [_eda_save_figure(make_channel_figure(conv_rows, title), folder / "channels_overview")]
    page_size = min(32, max(1, int(globals().get("EDA_MAX_LAYERS_PER_PAGE", 32))))
    for offset in range(0, len(conv_rows), page_size):
        page = offset // page_size + 1
        files.append(_eda_save_figure(make_channel_figure(conv_rows[offset:offset + page_size],
                     f"{title} — layers {offset + 1}–{min(offset + page_size, len(conv_rows))}"),
                     folder / f"channels_detail_{page:02d}"))
    fig, ax = _eda_plt.subplots(figsize=(max(14, min(32, len(conv_rows) * 0.16)), 5))
    ax.bar(range(len(conv_rows)), [row["out_channels_reduction_pct"] for row in conv_rows], color="red")
    ax.set_xticks(range(len(conv_rows)), [row["layer_name"] for row in conv_rows], rotation=90, fontsize=6)
    ax.set_ylabel("Output channels removed (%)")
    ax.set_title(title + " — physical Conv-output reduction")
    ax.set_ylim(0, 100)
    fig.tight_layout()
    files.append(_eda_save_figure(fig, folder / "channel_reduction_percent"))
    return {"status": "MEASURED", "conv_layers": len(conv_rows), "detail_page_size": page_size, "files": files}


def _eda_mapping_plots(audit, conv_rows, folder):
    mapping = audit.get("channel_mapping")
    if not mapping:
        return {"status": "UNAVAILABLE", "reason": "Original channel indices were not recorded; no reconstruction attempted."}
    _eda_write_json(folder / "channel_mapping.json", mapping)
    _eda_write_csv(folder / "channel_mapping.csv", mapping)
    masks = audit.get("channel_masks")
    if masks:
        _eda_write_json(folder / "channel_masks.json", masks)
    by_layer = {row["layer"]: row for row in mapping if row.get("module_type") == "Conv2d"
                and row.get("dimension") in {"out", "out_channels", "output", "output_channels"}}
    selected = []
    for row in conv_rows:
        item = by_layer.get(row["layer_name"])
        if item is None:
            return {"status": "UNAVAILABLE", "reason": "Mapping does not cover every Conv output.", "layer": row["layer_name"]}
        before = int(item["channels_before"])
        kept, removed = item.get("kept_original_indices"), item.get("removed_original_indices")
        if not isinstance(kept, list) or not isinstance(removed, list):
            raise TypeError(f"Missing original-index ledger for {row['layer_name']}.")
        if (len(set(kept)) != len(kept) or len(set(removed)) != len(removed)
                or set(kept) & set(removed) or set(kept) | set(removed) != set(range(before))
                or len(kept) != item["channels_after"]
                or before != row["out_channels_before"] or len(kept) != row["out_channels_after"]):
            raise ValueError(f"Original-index ledger disagrees with measured channels at {row['layer_name']}.")
        selected.append((row["layer_name"], before, set(removed)))
    page_size = min(32, max(1, int(globals().get("EDA_MAX_LAYERS_PER_PAGE", 32))))
    files = []
    for offset in range(0, len(selected), page_size):
        page = selected[offset:offset + page_size]
        width = max(item[1] for item in page)
        data = _eda_np.full((len(page), width), _eda_np.nan)
        for index, (_name, count, removed) in enumerate(page):
            data[index, :count] = 0
            if removed:
                data[index, sorted(removed)] = 1
        fig, ax = _eda_plt.subplots(figsize=(16, max(4, len(page) * 0.25)))
        cmap = _EdaColors(["#4477cc", "#ee3333"]).with_extremes(bad="#eeeeee")
        plot = ax.imshow(data, interpolation="nearest", aspect="auto", cmap=cmap, vmin=0, vmax=1)
        ax.set_yticks(range(len(page)), [item[0] for item in page], fontsize=7)
        ax.set_xlabel("Original Conv-output channel index (before any pruning)")
        ax.set_title("Original-index mask: blue kept, red removed, grey outside layer width")
        fig.colorbar(plot, ax=ax, ticks=[0, 1]).ax.set_yticklabels(["Kept", "Removed"])
        fig.tight_layout()
        files.append(_eda_save_figure(fig, folder / f"channel_mask_{offset // page_size + 1:02d}"))
    return {"status": "MEASURED", "conv_layers": len(selected), "files": files,
            "meaning": "Physical Conv-output masks, including coupled dimensions; never summed as independent channels."}


def _eda_lamp_plots(audit, folder):
    scores = audit.get("lamp_scores")
    if not scores:
        return {"status": "UNAVAILABLE", "reason": "Finite pre-cap LAMP scores were not recorded."}
    if any(not _eda_finite(row.get("score")) for row in scores):
        raise ValueError("LAMP EDA requires finite original scores; capacity sentinels must not be plotted.")
    _eda_write_csv(folder / "lamp_scores.csv", scores)
    _eda_write_json(folder / "lamp_scores.json", scores)
    fig, ax = _eda_plt.subplots(figsize=(10, 5))
    for removed, color, label in ((False, "blue", "Kept"), (True, "red", "Removed")):
        values = [row["score"] for row in scores if bool(row.get("removed")) == removed]
        if values:
            ax.hist(values, bins=min(50, max(5, int(len(values) ** 0.5))), alpha=0.55, color=color, label=label)
    ax.set_xlabel("Finite LAMP score before capacity mask")
    ax.set_ylabel("Independent root channels")
    ax.set_title("LAMP selection — descriptive score distribution")
    ax.legend()
    fig.tight_layout()
    files = [_eda_save_figure(fig, folder / "lamp_score_distribution")]
    capacities = {}
    for row in scores:
        group = capacities.setdefault(row["group_id"], {"channels": 0, "capacity_eligible": 0, "removed": 0})
        group["channels"] += 1
        group["capacity_eligible"] += bool(row.get("capacity_eligible"))
        group["removed"] += bool(row.get("removed"))
    capacity_rows = [{"group_id": key, **value} for key, value in capacities.items()]
    _eda_write_csv(folder / "lamp_group_capacity.csv", capacity_rows)
    if any(row["removed"] > row["capacity_eligible"] for row in capacity_rows):
        raise ValueError("LAMP selection removed channels outside the audited capacity mask.")
    fig, ax = _eda_plt.subplots(figsize=(max(12, min(24, len(capacity_rows) * .45)), 6))
    positions = _eda_np.arange(len(capacity_rows))
    blocked = [row["channels"] - row["capacity_eligible"] for row in capacity_rows]
    retained = [row["capacity_eligible"] - row["removed"] for row in capacity_rows]
    removed = [row["removed"] for row in capacity_rows]
    ax.bar(positions, blocked, color="grey", label="Protected by capacity cap")
    ax.bar(positions, retained, bottom=blocked, color="blue", label="Capacity eligible, kept")
    ax.bar(positions, removed, bottom=_eda_np.array(blocked) + retained, color="red", label="Removed")
    ax.set_xticks(positions, [row["group_id"] for row in capacity_rows], rotation=90, fontsize=7)
    ax.set_ylabel("Independent root channels")
    ax.set_title("Group capacity and actual LAMP selection")
    ax.legend()
    fig.tight_layout()
    files.append(_eda_save_figure(fig, folder / "lamp_group_capacity"))
    return {"status": "MEASURED", "finite_scores": len(scores),
            "capacity_eligible": sum(bool(row.get("capacity_eligible")) for row in scores),
            "removed": sum(bool(row.get("removed")) for row in scores), "files": files}


def _eda_efficiency_plots(stages, result, folder):
    files = []
    for metric, label in (("physical_conv_outputs", "Physical Conv-output channels"),
                          ("parameters", "Parameters (Conv + BN)"),
                          ("estimated_flops", "Estimated layer FLOPs (Conv + BN)")):
        fig, ax = _eda_plt.subplots(figsize=(14, 5))
        positions = _eda_np.arange(len(stages))
        ax.bar(positions - .2, [row[metric + "_before"] for row in stages], width=.4, color="blue", label="Before")
        ax.bar(positions + .2, [row[metric + "_after"] for row in stages], width=.4, color="red", label="After")
        ax.set_xticks(positions, [str(row["stage"]) for row in stages], rotation=90)
        ax.set_ylabel(label)
        ax.set_title("Measured stage inventory — " + label)
        ax.legend()
        fig.tight_layout()
        files.append(_eda_save_figure(fig, folder / ("stage_" + metric)))
    baseline, pruned = result.get("baseline", {}), result.get("pruned", {})
    summary = {"independent_eligible_channels_before": result.get("eligible_channels_before"),
               "independent_eligible_channels_after": result.get("eligible_channels_after"),
               "actual_independent_channel_ratio": result.get("actual_eligible_channel_ratio"),
               "physical_conv_outputs_before": sum(row["physical_conv_outputs_before"] for row in stages),
               "physical_conv_outputs_after": sum(row["physical_conv_outputs_after"] for row in stages),
               "baseline": {key: baseline.get(key) for key in ("parameters", "gflops", "checkpoint_bytes", "latency")},
               "pruned": {key: pruned.get(key) for key in ("parameters", "gflops", "checkpoint_bytes", "latency")},
               "comparison": result.get("comparison", {}),
               "latency_scope": "raw model forward, not end-to-end",
               "layer_flops_scope": "Conv/BN estimate; model GFLOPs also counts functional attention"}
    _eda_write_json(folder / "efficiency_summary.json", summary)
    return {"status": "MEASURED", "files": files, **summary}


def _eda_evaluation_plots(result, folder):
    statuses, files = {}, []
    for identity in ("baseline", "pruned"):
        validation = (result.get(identity) or {}).get("validation")
        if not validation:
            statuses[identity] = {"status": "NOT_RUN", "reason": "Dataset validation was not run."}
            continue
        target = folder / identity
        target.mkdir(parents=True, exist_ok=True)
        per_class = validation.get("per_class") or []
        if per_class:
            _eda_write_csv(target / "per_class_metrics.csv", per_class)
            labels = [row.get("name", str(row.get("class_id"))) for row in per_class]
            fig, axes = _eda_plt.subplots(1, 3, figsize=(21, 5))
            for key, label in (("ap50", "AP50"), ("ap50_95", "AP50–95")):
                indices = [index for index, row in enumerate(per_class) if _eda_finite(row.get(key))]
                if indices:
                    axes[0].plot(indices, [per_class[index][key] for index in indices], "o-", label=label)
            for key, label in (("precision", "Precision"), ("recall", "Recall"), ("f1", "F1")):
                indices = [index for index, row in enumerate(per_class) if _eda_finite(row.get(key))]
                if indices:
                    axes[1].plot(indices, [per_class[index][key] for index in indices], "o-", label=label)
            support_indices = [index for index, row in enumerate(per_class) if _eda_finite(row.get("support"))]
            if support_indices:
                axes[2].bar(support_indices, [per_class[index]["support"] for index in support_indices], color="grey")
            else:
                axes[2].text(.5, .5, "Class support unavailable", ha="center", va="center", transform=axes[2].transAxes)
            for ax in axes:
                ax.set_xticks(range(len(labels)), labels, rotation=45, ha="right")
                if ax.lines:
                    ax.legend()
            for ax in axes[:2]:
                ax.set_ylim(0, 1)
                ax.set_ylabel("Measured metric")
            axes[2].set_ylabel("Ground-truth instance support")
            fig.suptitle(identity + " — measured class metrics; missing values are omitted")
            fig.tight_layout()
            files.append(_eda_save_figure(fig, target / "class_metrics"))
        curves = validation.get("curves")
        if curves:
            _eda_write_json(target / "curves.json", curves)
        curve_status = _eda_curve_plots(curves, target) if curves else {"status": "UNAVAILABLE"}
        confusion = validation.get("confusion_matrix") or {}
        confusion_status = {"status": "UNAVAILABLE", "reason": "No measured confusion matrix was collected."}
        if (confusion.get("measured") is True or confusion.get("status") == "MEASURED") and confusion.get("matrix") is not None:
            matrix = _eda_np.asarray(confusion["matrix"], dtype=float)
            labels = confusion.get("labels", [])
            if matrix.ndim != 2 or matrix.shape != (len(labels), len(labels)) or not _eda_np.isfinite(matrix).all():
                raise ValueError("Confusion matrix dimensions/labels/values are invalid.")
            _eda_write_json(target / "confusion_matrix.json", confusion)
            _eda_write_csv(target / "confusion_matrix.csv", [
                {"predicted_class": labels[index], **{label: float(value) for label, value in zip(labels, row)}}
                for index, row in enumerate(matrix)])
            fig, ax = _eda_plt.subplots(figsize=(9, 8))
            plotted = ax.imshow(matrix, cmap="Blues")
            ax.set_xticks(range(len(labels)), labels, rotation=45, ha="right")
            ax.set_yticks(range(len(labels)), labels)
            ax.set_xlabel(confusion.get("x_label", "True class"))
            ax.set_ylabel(confusion.get("y_label", "Predicted class"))
            conf = confusion.get("confidence_threshold", confusion.get("conf"))
            matching_iou = confusion.get("matching_iou_threshold", confusion.get("iou"))
            ax.set_title(f"{identity}: measured confusion matrix\nconf={conf}, matching IoU={matching_iou}")
            fig.colorbar(plotted, ax=ax)
            fig.tight_layout()
            files.append(_eda_save_figure(fig, target / "confusion_matrix"))
            confusion_status = {"status": "MEASURED", "confidence_threshold": conf, "matching_iou_threshold": matching_iou,
                                "validation_iou": confusion.get("validation_iou")}
        statuses[identity] = {"status": "MEASURED", "per_class": "MEASURED" if per_class else "UNAVAILABLE",
                              "curves": curve_status, "confusion_matrix": confusion_status}
    return {"status": "MEASURED" if any(row["status"] == "MEASURED" for row in statuses.values()) else "NOT_RUN",
            "measurements": statuses, "files": files,
            "interpretation": "Descriptive association; layer changes do not establish a cause of mAP changes."}


def _eda_curve_plots(curves, folder):
    entries = curves if isinstance(curves, list) else curves.get("curves", curves.get("items", []))
    count, files = 0, []
    for entry in entries:
        if not isinstance(entry, dict) or entry.get("status") in {"UNAVAILABLE", "NOT_RUN"}:
            continue
        x, y = entry.get("x"), entry.get("y")
        if x is None or y is None:
            continue
        xs, ys = _eda_np.asarray(x, dtype=float), _eda_np.asarray(y, dtype=float)
        if xs.ndim != 1 or not xs.size or not _eda_np.isfinite(xs).all() or not _eda_np.isfinite(ys).all():
            continue
        if ys.ndim == 1:
            ys = ys[None, :]
        if ys.ndim != 2 or ys.shape[-1] != len(xs):
            continue
        fig, ax = _eda_plt.subplots(figsize=(10, 6))
        labels = entry.get("labels") or entry.get("class_names") or []
        for index, values in enumerate(ys):
            ax.plot(xs, values, label=str(labels[index]) if index < len(labels) else f"series {index}", alpha=.75)
        ax.set_xlabel(entry.get("x_label", entry.get("xlabel", "x")))
        ax.set_ylabel(entry.get("y_label", entry.get("ylabel", "y")))
        ax.set_title(entry.get("name", "Measured validation curve"))
        if len(ys) <= 20:
            ax.legend()
        fig.tight_layout()
        files.append(_eda_save_figure(fig, folder / f"evaluation_curve_{count:02d}"))
        count += 1
    return {"status": "MEASURED" if count else "UNAVAILABLE", "curve_count": count, "files": files}


def _build_run_eda(result, run_dir, inventory_before, inventory_after):
    """Export ordered EDA from actual run records; failures propagate to run reports."""
    folder = _EdaPath(run_dir) / "eda"
    folder.mkdir(parents=True, exist_ok=True)
    rows = layer_eda_rows(inventory_before, inventory_after)
    conv_rows = [row for row in rows if row["module_type"] == "Conv2d"]
    stages = stage_eda_rows(rows)
    _eda_write_csv(folder / "layer_eda.csv", rows)
    _eda_write_csv(folder / "stage_eda.csv", stages)
    audit = result.get("audit") or {}
    title = f"{result.get('model_name', 'YOLO26' + result.get('model_variant', 'n'))} {result.get('policy', '')} pr{result.get('ratio', 0):g}"
    report = {"status": "MEASURED", "model_variant": result.get("model_variant", "n"),
              "model_name": result.get("model_name", "YOLO26n"), "policy": result.get("policy"), "ratio": result.get("ratio"),
              "section_order": ["identity_coverage", "channels", "masks_lamp", "efficiency", "validation_tradeoffs"],
              "identity_coverage": {"provenance": result.get("provenance"), "conv_layers": len(conv_rows),
                                    "bn_layers": sum(row["module_type"] == "BatchNorm2d" for row in rows),
                                    "eligible_layers": sum(bool(row["eligible"]) for row in rows),
                                    "protected_layers": sum(bool(row["protection_reason"]) for row in rows),
                                    "group_count": result.get("pruned_group_count"),
                                    "protection_reasons": sorted({str(row["protection_reason"]) for row in rows if row["protection_reason"]})},
              "channels": _eda_channel_plots(conv_rows, folder, title),
              "channel_mapping": _eda_mapping_plots(audit, conv_rows, folder),
              "lamp_scores": _eda_lamp_plots(audit, folder),
              "efficiency": _eda_efficiency_plots(stages, result, folder),
              "validation_tradeoffs": _eda_evaluation_plots(result, folder),
              "notes": ["Channel figures include all Conv outputs in original module order.",
                        "Independent root-channel counts differ from physical coupled Conv-output changes.",
                        "No activation feature maps, simulated accuracy, or reconstructed historical channel indices."]}
    _eda_write_json(folder / "eda_status.json", report)
    return report


def build_run_eda(result, run_dir, inventory_before, inventory_after):
    """Persist an explicit failed EDA report before propagating analysis errors."""
    try:
        return _build_run_eda(result, run_dir, inventory_before, inventory_after)
    except Exception as error:
        _eda_write_json(_EdaPath(run_dir) / "eda/eda_status.json",
                        {"status": "FAIL", "model_variant": result.get("model_variant", "n"),
                         "policy": result.get("policy"), "ratio": result.get("ratio"),
                         "error": {"type": type(error).__name__, "message": str(error)}})
        raise


def plot_model_comparison(suites, output_dir):
    """Compare N/S only where each point has measured accuracy and efficiency."""
    folder = _EdaPath(output_dir)
    folder.mkdir(parents=True, exist_ok=True)
    suites = list(suites.values()) if isinstance(suites, dict) else list(suites)
    points = []
    for suite in suites:
        variant = suite.get("model_variant", "n")
        name = suite.get("model_name", "YOLO26" + variant)
        baseline = suite.get("baseline") or {}
        for ratio, policy, metrics in [(0., "baseline", baseline)] + [
                (run["ratio"], run.get("policy", "extended"), run.get("pruned") or {})
                for run in suite.get("results", [])]:
            validation = metrics.get("validation") or {}
            latency = metrics.get("latency") or {}
            points.append({"model_variant": variant, "model_name": name, "ratio": ratio, "policy": policy,
                           "map50_95": validation.get("map50_95"), "parameters": metrics.get("parameters"),
                           "gflops": metrics.get("gflops"),
                           "latency_ms": latency.get("mean_ms") if latency.get("status") in {"PASS", "MEASURED"} else None,
                           "latency_p05_ms": latency.get("p05_ms") if latency.get("status") in {"PASS", "MEASURED"} else None,
                           "latency_p95_ms": latency.get("p95_ms") if latency.get("status") in {"PASS", "MEASURED"} else None})
    _eda_write_csv(folder / "model_comparison.csv", points)
    statuses = []
    for metric, label in (("parameters", "Parameters"), ("gflops", "GFLOPs"), ("latency_ms", "Raw model forward latency (ms)")):
        valid = [row for row in points if _eda_finite(row.get(metric)) and _eda_finite(row.get("map50_95"))]
        if not valid:
            statuses.append({"plot": "map_vs_" + metric, "status": "NOT_RUN", "reason": "Paired measured accuracy/efficiency is unavailable."})
            continue
        fig, ax = _eda_plt.subplots(figsize=(10, 7))
        for variant in dict.fromkeys(row["model_variant"] for row in valid):
            rows = [row for row in valid if row["model_variant"] == variant]
            ax.scatter([row[metric] for row in rows], [row["map50_95"] for row in rows], label="YOLO26" + variant)
            for row in rows:
                if (metric == "latency_ms" and _eda_finite(row.get("latency_p05_ms"))
                        and _eda_finite(row.get("latency_p95_ms"))):
                    ax.hlines(row["map50_95"], row["latency_p05_ms"], row["latency_p95_ms"], alpha=.5)
                ax.annotate(f"{row['model_name']} {row['policy']} pr{row['ratio']:g}", (row[metric], row["map50_95"]),
                            xytext=(5, 5), textcoords="offset points", fontsize=8)
        ax.set_xlabel(label)
        ax.set_ylabel("Measured mAP50–95")
        ax.set_title("N/S compression–accuracy comparison")
        ax.legend()
        fig.tight_layout()
        files = _eda_save_figure(fig, folder / ("map_vs_" + metric))
        statuses.append({"plot": "map_vs_" + metric, "status": "MEASURED", "points": len(valid), "files": files})
    report = {"plots": statuses, "point_count": len(points), "scope": "Each model uses its own measured baseline; no missing accuracy is replaced with zero."}
    _eda_write_json(folder / "comparison_status.json", report)
    return report

## 07 — Implementasi LAMP Structured Pruning
`LAMPImportance(p=2, group_reduction="mean", bias=False)` dipakai oleh BasePruner.
Graph dibangun pada eval dengan autograd aktif, memakai raw outputs kedua cabang; tanpa backward.
Cap 50% dibatasi sebelum ranking menggunakan sentinel +inf bagi kandidat di luar kapasitas;
skor LAMP kandidat tetap asli. Setiap proposal diaudit ulang sebelum penghapusan fisik.
Pruning bukan masking. Operasi tidak dikenal menghentikan proposal terpilih.

## 08 — Eksperimen Independen dan Fine-tuning Recovery
Setiap konfigurasi memuat baseline asli dan memverifikasi state hash sebelum pruning.
Baseline diukur/divalidasi sekali per model; checkpoint pruned asli tetap disimpan sebagai
pembanding sebelum recovery. Trainer mempertahankan objek DetectionModel struktural dan
memeriksa state/shape awal agar YAML standar tidak membangun ulang channel yang dihapus.

Recovery menjalankan training biasa pada split train dengan loss deteksi native, optimizer
baru dan seed yang dicatat. Validation memilih best EMA. Semua bobot yang dapat dilatih,
termasuk head, dapat diperbarui selama recovery; perlindungan head berarti dimensi tidak
dipangkas. Dimensi hasil pruning wajib tetap sama sebelum/sesudah fine-tuning.
Dataset tidak tersedia menghasilkan NOT_RUN tanpa download pengganti; dataset invalid
atau kegagalan training disimpan ke laporan lalu exception diteruskan.

## 09 — Checkpoint Pruned, Recovered, dan Fresh Reload
Checkpoint sebelum recovery dipertahankan sebagai `pruned_fp32.pt`; checkpoint hasil
best EMA disimpan sebagai `recovery/recovered_fp32.pt`. Keduanya memuat objek DetectionModel
native struktural FP32 dengan `ema=None`, `optimizer=None`, tanpa kelas trainer khusus.
Checkpoint training best/last dan laporan epoch disimpan terpisah.

Fresh Python tanpa import repository memuat YOLO(checkpoint), menjalankan kedua cabang,
prediction, dan parity FP32 raw/decoded outputs (`rtol=1e-4`, `atol=1e-5`). Bentuk tensor,
jumlah parameter dan class order wajib cocok dengan model pruned; akurasi dinilai terpisah.

In [ ]:
RELOAD_SCRIPT = r'''
import json, sys
from pathlib import Path
import numpy as np
import torch
from ultralytics import YOLO

checkpoint, reference, result, device = sys.argv[1:]
torch.set_num_threads(2)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision("highest")
payload = torch.load(reference, map_location="cpu", weights_only=True)
wrapper = YOLO(checkpoint, task="detect")
core = wrapper.model.to(device).float().eval()
with torch.no_grad():
    output = core(payload["image"].to(device))
snapshot = {"decoded": output[0].detach().cpu()}
for branch in ("one2many", "one2one"):
    for key in ("boxes", "scores"):
        snapshot[branch + "/" + key] = output[1][branch][key].detach().cpu()
for key, expected in payload["expected"].items():
    if not torch.isfinite(snapshot[key]).all():
        raise RuntimeError("Non-finite output after reload: " + key)
    torch.testing.assert_close(snapshot[key], expected, rtol=1e-4, atol=1e-5)
if sum(p.numel() for p in core.parameters()) != payload["parameters"]:
    raise RuntimeError("Parameter count changed after reload")
if not all(p.dtype == torch.float32 for p in core.parameters()):
    raise RuntimeError("Reload is not FP32")
torch.save(snapshot, result)
# Fresh wrapper protects the raw unfused model from inference backend fusion.
predictor = YOLO(checkpoint, task="detect")
size = payload["image"].shape[-1]
predictions = predictor.predict(source=np.zeros((size, size, 3), dtype=np.uint8),
    imgsz=size, device=device, conf=0.001, iou=0.7, max_det=300,
    quantize=None, nms=None, verbose=False, save=False)
if len(predictions) != 1 or predictions[0].boxes is None:
    raise RuntimeError("No valid detection Results after reload")
boxes = predictions[0].boxes.data.detach().cpu()
if boxes.ndim != 2 or boxes.shape[1] != 6 or not torch.isfinite(boxes).all():
    raise RuntimeError("Invalid detection tensor")
nc = len(core.names)
if boxes.numel() and (((boxes[:, 5] < 0) | (boxes[:, 5] >= nc) | (boxes[:, 5] != boxes[:, 5].floor())).any()
                     or ((boxes[:, 4] < 0) | (boxes[:, 4] > 1)).any()):
    raise RuntimeError("Invalid detection class/confidence")
Path(result).with_suffix(".json").write_text(json.dumps({"prediction_count": len(boxes),
    "parameters": payload["parameters"], "direct_yolo_load": True,
    "raw_parity": True, "prediction_shape": list(boxes.shape)}), encoding="utf-8")
'''


def save_fp32_checkpoint(core, path, provenance):
    """Simpan arsitektur hasil pruning memakai format model-object Ultralytics."""
    model = deepcopy(core).cpu().float().eval().requires_grad_(False)
    for module in model.modules():
        module._forward_hooks.clear()
        module._forward_pre_hooks.clear()
        module._backward_hooks.clear()
    model.args = {"task": "detect", "imgsz": IMGSZ, "data": str(DATA_YAML), "nms": None, "quantize": None}
    torch.save({"model": model, "ema": None, "optimizer": None, "epoch": -1,
                "train_args": dict(model.args), "version": ultralytics.__version__,
                "date": datetime.now(timezone.utc).isoformat(), "lamp_format_version": 1,
                "lamp_provenance": provenance, "recovery_enabled": False}, path)
    return {"path": str(Path(path).resolve()), "sha256": sha256_file(path), "file_bytes": Path(path).stat().st_size}


def verify_fresh_reload(checkpoint, core, image, output_dir, *, timeout=300):
    """Restore di interpreter baru; exit code/error tidak boleh disembunyikan."""
    output_dir = Path(output_dir).resolve()
    script = output_dir / "verify_reload.py"
    reference = output_dir / "reload_reference.pt"
    result = output_dir / "reload_output.pt"
    script.write_text(RELOAD_SCRIPT, encoding="utf-8")
    expected = forward_snapshot(core, image)
    torch.save({"image": image.detach().cpu(), "expected": expected,
                "parameters": sum(p.numel() for p in core.parameters())}, reference)
    child_env = dict(os.environ)
    child_env.pop("PYTHONPATH", None)
    child_env["WANDB_MODE"] = "disabled"
    child_env["YOLO_AUTOINSTALL"] = "false"
    completed = subprocess.run([sys.executable, "-B", str(script), str(Path(checkpoint).resolve()),
                                str(reference), str(result), str(next(core.parameters()).device)],
                               cwd=output_dir, env=child_env, capture_output=True, text=True,
                               encoding="utf-8", errors="replace", timeout=timeout)
    (output_dir / "reload_stdout.txt").write_text(completed.stdout, encoding="utf-8")
    (output_dir / "reload_stderr.txt").write_text(completed.stderr, encoding="utf-8")
    if completed.returncode:
        raise RuntimeError(f"Fresh-process reload gagal (exit {completed.returncode}): {completed.stderr[-6000:]}")
    actual = torch.load(result, map_location="cpu", weights_only=True)
    for key in expected:
        torch.testing.assert_close(actual[key], expected[key], rtol=1e-4, atol=1e-5)
    report = json.loads(result.with_suffix(".json").read_text(encoding="utf-8"))
    report["max_absolute_difference"] = {key: float((actual[key] - expected[key]).abs().max()) for key in expected}
    report["process"] = "fresh Python, no repository PYTHONPATH"
    return report

## 10 — Validasi Baseline, Pruned, dan Recovered
Ketiga tahap memakai protokol identik: val, 640, confidence 0.001, IoU 0.7, max_det 300,
nms=None, FP32 dan tanpa TTA. Class order wajib sama. Dataset tersedia tetapi invalid
menghasilkan FAIL; dataset tidak tersedia menghasilkan NOT_RUN. Tidak memakai fallback V2.
Latency CUDA berupa raw unfused model forward, batch satu, 50 warm-up, 200 pengukuran
dan sinkronisasi; preprocessing/NMS/disk IO tidak termasuk.

Bandingkan penurunan akurasi akibat pruning, perubahan akibat fine-tuning, dan selisih akhir
terhadap baseline. Status pipeline dan besar pemulihan akurasi dilaporkan secara terpisah.

In [ ]:
def prepare_validation_dataset(data_yaml, dataset_root, output_dir):
    """Validasi dataset lokal tanpa download atau normalisasi label otomatis."""
    source = Path(data_yaml).expanduser()
    if not source.is_file():
        return None
    raw = yaml.safe_load(source.read_text(encoding="utf-8"))
    if not isinstance(raw, dict) or not raw.get("val") or not raw.get("names"):
        raise ValueError("DATA_YAML tersedia tetapi tidak berisi val/names yang valid.")
    raw.pop("download", None)
    root = Path(dataset_root).expanduser().resolve() if dataset_root is not None else source.resolve().parent
    if not root.is_dir():
        raise FileNotFoundError(f"DATASET_ROOT tidak tersedia: {root}")
    raw["path"] = str(root)
    copy_path = Path(output_dir) / "evaluation_data.yaml"
    copy_path.write_text(yaml.safe_dump(raw, sort_keys=False, allow_unicode=True), encoding="utf-8")
    info = check_det_dataset(str(copy_path), autodownload=False)
    splits = info["val"] if isinstance(info["val"], list) else [info["val"]]
    if any(not Path(split).exists() for split in splits):
        raise FileNotFoundError("Validation split tidak tersedia.")
    return {"yaml": str(copy_path.resolve()), "info": info, "source_yaml": str(source.resolve()),
            "source_yaml_sha256": sha256_file(source), "evaluation_yaml_sha256": sha256_file(copy_path),
            "val_paths": [str(split) for split in splits], "class_names": ordered_names(info["names"])}


def validation_arguments(data_yaml, output_dir, label):
    """Satu sumber pengaturan evaluasi untuk baseline dan pruned model."""
    return {"data": str(data_yaml), "split": "val", "imgsz": IMGSZ, "batch": BATCH,
            "workers": WORKERS, "device": str(TARGET_DEVICE), "conf": 0.001,
            "iou": 0.7, "max_det": 300, "quantize": None, "nms": None,
            "augment": False, "plots": False, "verbose": False, "save_json": False,
            "project": str(output_dir), "name": label, "exist_ok": False}


def evaluate_checkpoint(path, dataset, output_dir, label):
    """Evaluasi wrapper baru agar fusion validator tidak mengubah model pengukuran."""
    wrapper = YOLO(str(path), task="detect")
    assert_class_match(wrapper.names, dataset["info"]["names"])
    metric = wrapper.val(**validation_arguments(dataset["yaml"], output_dir, label))
    box = metric.box
    per_class = [{"class_id": i, "name": name, "ap50": None, "ap50_95": None}
                 for i, name in enumerate(dataset["class_names"])]
    for row, class_id in enumerate(box.ap_class_index):
        per_class[int(class_id)].update({"ap50": float(box.ap50[row]), "ap50_95": float(box.ap[row])})
    result = {"map50_95": float(box.map), "map50": float(box.map50),
              "precision": float(box.mp), "recall": float(box.mr), "per_class": per_class,
              "split": "val", "precision_mode": "FP32", "nms": None}
    if any(not math.isfinite(result[key]) for key in ("map50_95", "map50", "precision", "recall")):
        raise ValueError("Metrik evaluasi mengandung NaN/Inf.")
    return result


def cuda_latency(core, image, warmup=50, repeats=200):
    """Latency forward model batch satu; tidak mencakup preprocessing/NMS/disk IO."""
    if image.device.type != "cuda":
        return {"status": "NOT_RUN", "reason": "CUDA tidak tersedia; tidak mengklaim benchmark GPU."}
    core = deepcopy(core).eval().requires_grad_(False)
    samples = []
    with torch.inference_mode():
        for _ in range(warmup):
            core(image)
        torch.cuda.synchronize(image.device)
        for _ in range(repeats):
            torch.cuda.synchronize(image.device)
            started = time.perf_counter()
            core(image)
            torch.cuda.synchronize(image.device)
            samples.append((time.perf_counter() - started) * 1000)
    return {"status": "PASS", "median_ms": float(np.median(samples)), "mean_ms": float(np.mean(samples)),
            "p05_ms": float(np.percentile(samples, 5)), "p95_ms": float(np.percentile(samples, 95)),
            "warmup": warmup, "repeats": repeats, "batch": 1, "imgsz": list(image.shape[-2:]),
            "precision": "FP32", "device": str(image.device), "fusion": False, "scope": "raw model forward"}

In [ ]:
"""Validation declarations embedded verbatim in standalone LAMP V2 notebooks.

The source never imports repository modules at notebook runtime. Numerical exports
come from the actual Ultralytics validator; missing measurements remain null.
"""
# ruff: noqa: F821 -- Notebook configuration/helpers provide these runtime globals.


def validation_arguments(data_yaml, output_dir, label):
    """Identical FP32 protocol for baseline/pruned; plots also collect confusion counts."""
    return {"data": str(data_yaml), "split": "val", "imgsz": IMGSZ, "batch": BATCH,
            "workers": WORKERS, "device": str(TARGET_DEVICE), "conf": 0.001,
            "iou": 0.7, "max_det": 300, "quantize": None, "nms": None,
            "augment": False, "plots": True, "verbose": False, "save_json": False,
            "project": str(output_dir), "name": label, "exist_ok": False}


def _validation_number(value):
    """JSON scalar: unsupported/nonfinite auxiliary metrics are unavailable, not zero."""
    if value is None:
        return None
    try:
        number = float(value)
    except (TypeError, ValueError, OverflowError):
        return None
    return number if math.isfinite(number) else None


def _validation_array(value):
    import numpy as np

    if hasattr(value, "detach"):
        value = value.detach().cpu().numpy()
    return np.asarray(value, dtype=float)


def _validation_json_array(value):
    import numpy as np

    array = _validation_array(value)
    return np.where(np.isfinite(array), array, None).tolist()


def _validation_per_class(metric, class_names):
    """Map SDK result rows via ap_class_index, never by names or array position."""
    import numpy as np

    box = metric.box
    support = getattr(metric, "nt_per_class", None)
    if support is not None:
        support = _validation_array(support)
        if (support.shape != (len(class_names),) or not np.isfinite(support).all()
                or (support < 0).any() or not np.equal(support, np.floor(support)).all()):
            raise ValueError("Support kelas SDK tidak valid atau tidak cocok dengan jumlah kelas.")
    rows = [{"class_id": i, "name": name, "ap50": None, "ap50_95": None,
             "precision": None, "recall": None, "f1": None,
             "support": int(support[i]) if support is not None else None}
            for i, name in enumerate(class_names)]
    indices = list(getattr(box, "ap_class_index", []))
    validated_indices = []
    for value in indices:
        scalar = _validation_number(value)
        if scalar is None or scalar != int(scalar) or not 0 <= int(scalar) < len(rows):
            raise ValueError("Indeks kelas SDK tidak valid.")
        validated_indices.append(int(scalar))
    if len(set(validated_indices)) != len(validated_indices):
        raise ValueError("Indeks kelas SDK duplikat.")
    for offset, class_id in enumerate(validated_indices):
        for key, attribute in (("ap50", "ap50"), ("ap50_95", "ap"),
                               ("precision", "p"), ("recall", "r"), ("f1", "f1")):
            values = getattr(box, attribute, None)
            if values is not None and len(values) > offset:
                rows[class_id][key] = _validation_number(values[offset])
    return rows, validated_indices


def _validation_curves(metric, class_names, class_ids):
    """Keep SDK axes and class mapping, replacing nonfinite samples with JSON null."""
    import numpy as np

    try:
        curve_values = metric.curves_results
    except (AttributeError, TypeError, ValueError) as error:
        return {"status": "UNAVAILABLE", "reason": str(error), "curves": []}
    names = getattr(metric, "curves", [])
    records = []
    for index, values in enumerate(curve_values):
        record = {"name": names[index] if index < len(names) else f"SDK curve {index}",
                  "class_ids": list(class_ids), "class_names": [class_names[i] for i in class_ids],
                  "status": "UNAVAILABLE", "x": [], "y": []}
        try:
            x_value, y_value, x_label, y_label = values
            x, y = _validation_array(x_value), _validation_array(y_value)
            record.update({"x_label": str(x_label), "y_label": str(y_label)})
            if y.ndim == 1 and len(class_ids) == 1:
                y = y[None, :]
            if (not class_ids or x.ndim != 1 or not x.size or y.ndim != 2
                    or y.shape != (len(class_ids), x.size)):
                record["reason"] = "Dimensi kurva tidak memiliki pemetaan kelas SDK yang valid."
            else:
                record.update({"x": _validation_json_array(x), "y": _validation_json_array(y),
                               "status": "MEASURED" if np.isfinite(x).all() and np.isfinite(y).all()
                               else "PARTIAL"})
                if record["status"] == "PARTIAL":
                    record["reason"] = "Sampel SDK NaN/Inf disimpan sebagai null."
        except (TypeError, ValueError, OverflowError) as error:
            record["reason"] = str(error)
        records.append(record)
    states = {item["status"] for item in records}
    status = "MEASURED" if states == {"MEASURED"} else (
        "PARTIAL" if states & {"MEASURED", "PARTIAL"} else "UNAVAILABLE")
    return {"status": status, "curves": records}


def _validation_confusion_matrix(validator, class_names, arguments):
    """Export actual counts and the distinct matching/NMS IoU thresholds."""
    import inspect

    import numpy as np

    matrix_object = getattr(validator, "confusion_matrix", None)
    result = {"status": "UNAVAILABLE", "labels": [*class_names, "background"],
              "orientation": "rows=predicted, columns=true", "matrix": None,
              "normalized_by_true": None, "confidence_threshold": None,
              "matching_iou_threshold": None, "validation_iou": arguments["iou"],
              "normalization": "Divide each true-class column by its sum; empty columns stay zero."}
    if matrix_object is None or getattr(matrix_object, "matrix", None) is None:
        result["reason"] = "Validator tidak menyediakan confusion matrix."
        return result
    matrix = _validation_array(matrix_object.matrix)
    expected = len(class_names) + 1
    if (matrix.shape != (expected, expected) or not np.isfinite(matrix).all() or (matrix < 0).any()):
        raise ValueError("Confusion matrix SDK tidak valid atau class order berbeda.")
    threshold = _validation_number(getattr(validator, "confusion_matrix_conf", None))
    matching_iou = None
    try:
        parameter = inspect.signature(matrix_object.process_batch).parameters.get("iou_thres")
        if parameter is not None and parameter.default is not inspect.Parameter.empty:
            matching_iou = _validation_number(parameter.default)
    except (AttributeError, TypeError, ValueError):
        pass
    totals = matrix.sum(axis=0, keepdims=True)
    normalized = np.divide(matrix, totals, out=np.zeros_like(matrix), where=totals != 0)
    result.update({"status": "MEASURED", "matrix": matrix.tolist(),
                   "normalized_by_true": normalized.tolist(),
                   "zero_support_columns": [int(i) for i in np.flatnonzero(totals[0] == 0)],
                   "confidence_threshold": threshold, "matching_iou_threshold": matching_iou,
                   "matching_iou_source": "Signature default of actual ConfusionMatrix.process_batch; validator calls without iou_thres.",
                   "confidence_source": "Actual DetectionValidator.confusion_matrix_conf"})
    return result


def _validation_metric_semantics(metric):
    """Record the SDK best-F1 reporting point without confusing it with CM confidence."""
    import numpy as np
    from ultralytics.utils.metrics import smooth

    result = {"precision_recall_f1": "SDK values at the global confidence maximizing smoothed mean F1 (IoU=0.50).",
              "reporting_confidence": None,
              "reporting_confidence_source": "SDK px[smooth(f1_curve.mean(0), 0.1).argmax()]",
              "map_iou_thresholds": [round(0.5 + 0.05 * i, 2) for i in range(10)],
              "absent_class": "Support 0 denotes no ground truth; class metrics without an SDK row remain null.",
              "confusion_matrix": "A separate operating point: see confidence_threshold and matching_iou_threshold."}
    box = metric.box
    try:
        px, f1 = _validation_array(box.px), _validation_array(box.f1_curve)
        if (px.ndim == 1 and f1.ndim == 2 and f1.shape[0] > 0 and f1.shape[1] == len(px)
                and np.isfinite(px).all() and np.isfinite(f1).all()):
            result["reporting_confidence"] = float(px[int(smooth(f1.mean(0), 0.1).argmax())])
    except (AttributeError, TypeError, ValueError):
        pass
    return result


def evaluate_checkpoint(path, dataset, output_dir, label):
    """Evaluate a fresh wrapper; preserve SDK metrics, curves and populated confusion counts."""
    wrapper = YOLO(str(path), task="detect")
    assert_class_match(wrapper.names, dataset["info"]["names"])
    class_names = list(dataset["class_names"])
    assert_class_match(wrapper.names, class_names)
    arguments = validation_arguments(dataset["yaml"], output_dir, label)
    captured = {}
    capture_validator = None
    if hasattr(wrapper, "add_callback"):
        # Model.val() keeps its validator local in Ultralytics 8.4.155.
        # Read the actual instance at callback time rather than assuming an attribute.
        def capture_validator(validator):
            captured["validator"] = validator

        wrapper.add_callback("on_val_end", capture_validator)
    try:
        metric = wrapper.val(**arguments)
    finally:
        # Avoid a callback -> capture -> validator cycle retaining GPU allocations
        # between baseline/ratio runs. Removing it also affects the shared callbacks.
        callbacks = getattr(wrapper, "callbacks", {}).get("on_val_end", [])
        if capture_validator in callbacks:
            callbacks.remove(capture_validator)
    box = metric.box
    per_class, class_ids = _validation_per_class(metric, class_names)
    result = {"map50_95": _validation_number(box.map), "map50": _validation_number(box.map50),
              "precision": _validation_number(box.mp), "recall": _validation_number(box.mr),
              "per_class": per_class, "split": "val", "precision_mode": "FP32", "nms": None}
    if any(result[key] is None for key in ("map50_95", "map50", "precision", "recall")):
        raise ValueError("Metrik evaluasi mengandung NaN/Inf atau tidak tersedia.")
    result["curves"] = _validation_curves(metric, class_names, class_ids)
    validator = captured.get("validator", getattr(wrapper, "validator", None))
    result["confusion_matrix"] = _validation_confusion_matrix(validator, class_names, arguments)
    result["metric_semantics"] = _validation_metric_semantics(metric)
    export_dir = Path(output_dir) / label
    export_dir.mkdir(parents=True, exist_ok=True)
    write_csv(export_dir / "per_class_metrics.csv", per_class)
    write_json(export_dir / "curves.json", result["curves"])
    write_json(export_dir / "confusion_matrix.json", result["confusion_matrix"])
    write_json(export_dir / "metric_semantics.json", result["metric_semantics"])
    result["exports"] = {name: str(export_dir / name) for name in (
        "per_class_metrics.csv", "curves.json", "confusion_matrix.json", "metric_semantics.json")}
    return result

In [ ]:
def _init_tracking(result, run_dir):
    wandb_session_login()
    variant = result.get("model_variant", "n")
    config = configuration(result["policy"], result["ratio"], variant)
    config.update({"source_checkpoint_sha256": result["provenance"]["sha256"],
                   "dataset_yaml_sha256": result.get("dataset", {}).get("source_yaml_sha256"),
                   "versions": result["environment"]["packages"],
                   "validation_protocol": validation_arguments(DATA_YAML, run_dir, "protocol")})
    return wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, dir=str(run_dir), reinit="create_new",
                      config=config, **run_identity(result["policy"], result["ratio"], result["suite_id"], variant))


def tracking_metrics(result):
    before, after = result.get("baseline", {}), result.get("pruned", {})
    metric = {"pruning/target_ratio": result["ratio"],
              "smoke/structural_pass": int(result["structural_smoke_status"] == "PASS"),
              "smoke/validation_pass": int(result["validation_status"] == "PASS")}
    for label, values in (("baseline", before), ("pruned", after)):
        for key in ("parameters", "gflops", "checkpoint_bytes"):
            if values.get(key) is not None:
                metric[f"{label}/{key}"] = values[key]
        for key in ("map50_95", "map50", "precision", "recall"):
            value = (values.get("validation") or {}).get(key)
            if value is not None:
                metric[f"{label}/{key}"] = value
    for key in ("parameter_reduction_pct", "gflops_reduction_pct"):
        value = result.get("comparison", {}).get(key)
        if value is not None:
            metric["pruning/" + key] = value
    if result.get("actual_eligible_channel_ratio") is not None:
        metric["pruning/actual_eligible_ratio"] = result["actual_eligible_channel_ratio"]
    delta = result.get("comparison", {}).get("delta_map50_95")
    if delta is not None:
        metric["delta/map50_95"] = delta
        metric["delta/map50_95_pp"] = 100 * delta
    return metric


def track_result(result, run_dir):
    """Kegagalan jaringan tidak menghapus bukti lokal atau mengubah validitas struktur."""
    folder = Path(run_dir)
    report = {"status": "NOT_RUN", "uploads": [], "errors": []}
    run = None
    try:
        if not WANDB_TRACKING:
            report["reason"] = "WANDB_TRACKING=False"
            return report
        run = _init_tracking(result, folder)
        report["run_id"] = run.id
        report["run_url"] = run.url
        # Catat lineage sumber; hanya menggunakannya, tanpa mengubah artifact baseline.
        variant = result.get("model_variant", "n")
        run.use_artifact(model_spec(variant)["artifact"], type="model")
        run.log(tracking_metrics(result))
        run.summary.update({"policy": result["policy"], "ratio": result["ratio"],
                            "structural_smoke_status": result["structural_smoke_status"],
                            "validation_status": result["validation_status"],
                            "source_checkpoint_sha256": result["provenance"]["sha256"]})
        if UPLOAD_WANDB_ARTIFACTS:
            ratio_name = f"{result['ratio']:g}".replace(".", "p")
            stem = f"yolo26{variant}-lamp-v2-{result['policy']}-pr{ratio_name}-{result['suite_id']}"
            metadata_info = {"model_variant": variant, "policy": result["policy"], "ratio": result["ratio"], "seed": SEED,
                             "source": result["provenance"], "structural_smoke_status": result["structural_smoke_status"],
                             "validation_status": result["validation_status"]}
            for kind in ("model", "report"):
                artifact = wandb.Artifact(stem + "-" + kind, type=kind, metadata=metadata_info)
                if kind == "model":
                    if not (folder / "pruned_fp32.pt").is_file():
                        continue
                    artifact.add_file(str(folder / "pruned_fp32.pt"))
                    artifact.add_file(str(folder / "RESTORE.md"))
                else:
                    for path in sorted(folder.rglob("*")):
                        relative = path.relative_to(folder)
                        if (path.is_file() and relative.parts[0] != "wandb"
                                and path.suffix in {".json", ".csv", ".md", ".png", ".svg"}):
                            artifact.add_file(str(path), name=relative.as_posix())
                logged = run.log_artifact(artifact)
                logged.wait(timeout=300)
                report["uploads"].append({"type": kind, "status": "CONFIRMED", "name": logged.qualified_name})
        report["status"] = "PASS"
    except Exception as error:  # noqa: BLE001 -- Preserve local evidence for any SDK/network failure.
        report["status"] = "FAIL"
        report["errors"].append({"type": type(error).__name__, "message": safe_error(error)})
        print("W&B tracking/upload gagal; hasil lokal dipertahankan:", safe_error(error))
    finally:
        if run is not None:
            try:
                run.finish(exit_code=0 if report["status"] == "PASS" else 1)
            except Exception as error:  # noqa: BLE001 -- Finish failures must also be persisted locally.
                report["status"] = "FAIL"
                report["errors"].append({"type": type(error).__name__, "message": safe_error(error)})
        write_json(folder / "tracking_report.json", report)
    return report

In [ ]:
def _load_source_core(source_path, expected_hash):
    if sha256_file(source_path) != expected_hash:
        raise ValueError("Checkpoint sumber berubah antar konfigurasi.")
    return YOLO(str(source_path), task="detect").model.to(TARGET_DEVICE).float().eval()


def _prepare_baseline(source_path, provenance, suite_dir, model_variant="n"):
    folder = Path(suite_dir) / "baseline"
    core = _load_source_core(source_path, provenance["sha256"])
    audit = architecture_audit(core, model_variant)
    image = torch.zeros(1, 3, IMGSZ, IMGSZ, device=TARGET_DEVICE, dtype=torch.float32)
    forward_snapshot(core, image)
    measurements = model_measurements(core, IMGSZ)
    metadata_info = {"model_variant": model_variant, "model_name": f"YOLO26{model_variant}",
                     "state_sha256": model_state_sha256(core), "names": ordered_names(core.names),
                     "architecture": audit["report"]}
    payload = torch.load(source_path, map_location="cpu", weights_only=False)
    metadata_info["source_checkpoint_metadata"] = {
        key: payload.get(key) for key in ("epoch", "date", "version", "best_fitness")
        if isinstance(payload.get(key), (str, int, float, bool, type(None)))
    }
    source_args = payload.get("train_args", {}) or {}
    metadata_info["source_checkpoint_metadata"]["training"] = {
        key: source_args.get(key) for key in ("data", "model", "seed", "epochs", "imgsz", "task")
        if isinstance(source_args.get(key), (str, int, float, bool, type(None)))
    }
    del payload
    saved = save_fp32_checkpoint(core, folder / "baseline_fp32.pt", provenance)
    measurements["checkpoint_bytes"] = saved["file_bytes"]
    validation_report = {"status": "NOT_RUN", "reason": "dataset unavailable or validation disabled"}
    write_json(folder / "validation_report.json", validation_report)
    try:
        dataset = prepare_validation_dataset(DATA_YAML, DATASET_ROOT, folder) if RUN_FULL_VALIDATION else None
        if dataset is not None:
            assert_class_match(core.names, dataset["class_names"])
            measurements["validation"] = evaluate_checkpoint(folder / "baseline_fp32.pt", dataset, folder, "validation")
            validation_report = {"status": "PASS", "source_yaml_sha256": dataset["source_yaml_sha256"]}
    except Exception as error:
        write_json(folder / "validation_report.json", {"status": "FAIL", "type": type(error).__name__, "error": safe_error(error)})
        raise
    write_json(folder / "validation_report.json", validation_report)
    measurements["latency"] = (cuda_latency(core, image, LATENCY_WARMUP, LATENCY_REPEATS)
                                if RUN_LATENCY_BENCHMARK else {"status": "NOT_RUN", "reason": "disabled"})
    metadata_info.update({"saved_checkpoint": saved, "validation_status": "PASS" if dataset else "NOT_RUN",
                          "validation_reason": None if dataset else "dataset unavailable or validation disabled"})
    write_json(folder / "architecture_audit.json", audit["report"])
    write_json(folder / "metrics.json", measurements)
    write_json(folder / "metadata.json", metadata_info)
    return measurements, dataset, metadata_info


def _write_run_reports(folder, result):
    write_json(folder / "pruning_manifest.json", result)
    write_json(folder / "before_after_metrics.json", {
        "baseline": result["baseline"], "pruned": result["pruned"], "comparison": result["comparison"]})
    write_json(folder / "smoke_test_report.json", {key: result[key] for key in
               ("policy", "ratio", "checks", "structural_smoke_status", "validation_status", "stages", "errors", "warnings")})


def _run_configuration(source_path, provenance, folder, policy, ratio, baseline, dataset, baseline_metadata, environment, suite_id, model_variant="n"):
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=False)
    result = {"model_variant": model_variant, "model_name": f"YOLO26{model_variant}",
              "policy": policy, "ratio": ratio, "suite_id": suite_id,
              "structural_smoke_status": "NOT_RUN", "validation_status": "NOT_RUN",
              "checks": {key: False for key in REQUIRED_CHECKS}, "stages": {}, "errors": [], "warnings": [],
              "baseline": deepcopy(baseline), "pruned": {}, "comparison": {}, "provenance": provenance,
              "environment": environment, "dataset": {key: value for key, value in (dataset or {}).items() if key != "info"}}
    write_json(folder / "config.json", configuration(policy, ratio, model_variant))
    write_json(folder / "environment.json", environment)
    write_json(folder / "checkpoint_provenance.json", provenance)
    _write_run_reports(folder, result)
    stage = "baseline_reload"
    try:
        seed_environment(SEED)
        core = _load_source_core(source_path, provenance["sha256"])
        if model_state_sha256(core) != baseline_metadata["state_sha256"]:
            raise ValueError("Baseline awal berbeda: cumulative pruning atau sumber berubah.")
        result["initial_state_sha256"] = baseline_metadata["state_sha256"]
        assert_class_match(core.names, baseline_metadata["names"])
        result["checks"]["baseline_loaded"] = True
        image = torch.zeros(1, 3, IMGSZ, IMGSZ, device=TARGET_DEVICE, dtype=torch.float32)
        forward_snapshot(core, image)
        result["checks"]["baseline_forward"] = True
        protected = protected_policy_snapshot(core, model_variant)
        stage = "dependency_graph"
        pruner, audit = build_policy_pruner(core, image, policy, ratio, model_variant)
        result["checks"]["dependency_graph"] = True
        result["audit"] = audit["report"]
        write_json(folder / "dependency_audit.json", audit["report"])
        inventory_before = measure_layer_inventory(core, image, audit, policy)
        write_csv(folder / "layer_audit_before.csv", inventory_before)
        stage = "structural_pruning"
        history = apply_policy_pruning(pruner, audit)
        write_json(folder / "channel_mapping.json", audit["report"]["channel_mapping"])
        write_json(folder / "channel_masks.json", audit["report"]["channel_masks"])
        write_csv(folder / "lamp_scores.csv", audit["report"]["lamp_scores"])
        write_json(folder / "dependency_audit.json", audit["report"])
        result["groups"] = history
        result["pruned_group_count"] = len(history)
        result["checks"]["structural_channels_removed"] = bool(history)
        assert_layer_consistency(core)
        assert_policy_protected_unchanged(core, protected)
        result["checks"]["protected_head_unchanged"] = True
        forward_snapshot(core, image)
        result["checks"]["pruned_forward"] = True
        after = model_measurements(core, IMGSZ)
        if after["parameters"] >= baseline["parameters"]:
            raise ValueError("Parameter tidak berkurang setelah structural pruning.")
        result["checks"]["parameters_reduced"] = True
        initial = audit["independent_channels_before"]
        removed = sum(change["channels_before"] - change["channels_after"] for change in history)
        result["eligible_channels_before"] = initial
        result["eligible_channels_after"] = initial - removed
        result["actual_eligible_channel_ratio"] = removed / initial
        if abs(removed / initial - ratio) > 1e-12:
            result["warnings"].append("Rasio aktual berbeda karena integer channel atau threshold ranking; denominator tercatat eksplisit.")
        inventory_after = measure_layer_inventory(core, image, audit, policy)
        write_csv(folder / "layer_audit_after.csv", inventory_after)
        before_by_name = {row["layer_name"]: row for row in inventory_before}
        layer_changes = []
        for row in inventory_after:
            previous = before_by_name[row["layer_name"]]
            layer_changes.append({**row, "in_channels_before": previous["in_channels"],
                                  "out_channels_before": previous["out_channels"],
                                  "parameters_before": previous["parameters"],
                                  "estimated_flops_before": previous["estimated_flops"]})
        write_csv(folder / "layer_pruning_report.csv", layer_changes)
        result["physical_conv_output_channels_removed"] = sum(
            row["out_channels_before"] - row["out_channels"] for row in layer_changes if row["module_type"] == "Conv2d")
        stage = "save_and_reload"
        result["saved_pruned"] = save_fp32_checkpoint(core, folder / "pruned_fp32.pt", provenance)
        result["checks"]["checkpoint_saved"] = True
        (folder / "RESTORE.md").write_text(
            "# Restore LAMP V2 FP32\n\nGunakan versi runtime pada environment.json. "
            "Checkpoint berisi objek DetectionModel struktural; jangan muat state dict ke YAML standar.\n\n"
            "```python\nfrom ultralytics import YOLO\nmodel = YOLO('pruned_fp32.pt', task='detect')\n"
            "results = model.predict('image.jpg', imgsz=640, nms=None, quantize=None)\n```\n\n"
            "Tidak memerlukan import repository. Recovery belum dijalankan. "
            "Gunakan checkpoint dan budget identik untuk perbandingan recovery di masa depan.\n", encoding="utf-8")
        reload_report = verify_fresh_reload(folder / "pruned_fp32.pt", core, image, folder, timeout=RELOAD_TIMEOUT_SECONDS)
        reloaded = YOLO(str(folder / "pruned_fp32.pt"), task="detect").model.to(TARGET_DEVICE).float().eval()
        assert_class_match(reloaded.names, baseline_metadata["names"])
        assert_layer_consistency(reloaded)
        assert_policy_protected_unchanged(reloaded, protected)
        forward_snapshot(reloaded, image)
        result["reload"] = reload_report
        write_json(folder / "reload_report.json", reload_report)
        for key in ("checkpoint_reloaded", "reload_consistent", "detection_prediction"):
            result["checks"][key] = True
        if not all(result["checks"].values()):
            raise RuntimeError("Kontrak structural smoke belum terpenuhi.")
        result["structural_smoke_status"] = "PASS"
        after["checkpoint_bytes"] = result["saved_pruned"]["file_bytes"]
        result["pruned"] = after
        stage = "validation"
        if dataset is not None:
            after["validation"] = evaluate_checkpoint(folder / "pruned_fp32.pt", dataset, folder, "validation")
            result["validation_status"] = "PASS"
        else:
            result["warnings"].append("Validasi V3 NOT_RUN: dataset tidak tersedia atau dinonaktifkan.")
        stage = "latency"
        after["latency"] = (cuda_latency(reloaded, image, LATENCY_WARMUP, LATENCY_REPEATS)
                             if RUN_LATENCY_BENCHMARK else {"status": "NOT_RUN", "reason": "disabled"})
        result["comparison"] = metric_comparison(baseline, after)
        stage = "eda"
        result["eda"] = build_run_eda(result, folder, inventory_before, inventory_after)
        result["eda_status"] = "PASS"
        result["tradeoff_status"] = "MEASURED_NO_AUTOMATIC_PASS" if dataset else "NOT_ASSESSED"
        result["stages"]["experiment"] = "PASS"
    except Exception as error:
        result["stages"][stage] = "FAIL"
        result["errors"].append({"stage": stage, "type": type(error).__name__, "message": safe_error(error),
                                  "traceback": safe_error(traceback.format_exc())})
        if stage == "validation":
            result["validation_status"] = "FAIL"
        elif result["structural_smoke_status"] != "PASS":
            result["structural_smoke_status"] = "FAIL"
        else:
            result[stage + "_status"] = "FAIL"
        _write_run_reports(folder, result)
        result["tracking"] = track_result(result, folder)
        _write_run_reports(folder, result)
        raise
    _write_run_reports(folder, result)
    result["tracking"] = track_result(result, folder)
    _write_run_reports(folder, result)
    print(f"YOLO26{model_variant} {policy} pr{ratio:g}: structural={result['structural_smoke_status']}, "
          f"validation={result['validation_status']}, params={after['parameters']:,}, GFLOPs={after['gflops']:.6f}")
    return result


def summary_rows(suite):
    before = suite.get("baseline")
    if not before:
        return []
    entries = [("baseline", 0, before, {}, None)]
    entries.extend((r["policy"], r["ratio"], r["pruned"], r["comparison"], r) for r in suite.get("results", []))
    rows = []
    for policy, ratio, metric, comparison, result in entries:
        val = metric.get("validation") or {}
        variant = suite.get("model_variant", "n")
        rows.append({"model": f"YOLO26{variant}", "model_variant": variant, "policy": policy, "ratio": ratio,
                     "parameters": metric.get("parameters"), "gflops": metric.get("gflops"),
                     "checkpoint_bytes": metric.get("checkpoint_bytes"),
                     "map50_95": val.get("map50_95"), "map50": val.get("map50"),
                     "precision": val.get("precision"), "recall": val.get("recall"),
                     "parameter_reduction_pct": comparison.get("parameter_reduction_pct", 0),
                     "gflops_reduction_pct": comparison.get("gflops_reduction_pct", 0),
                     "delta_map50_95_pp": comparison.get("delta_map50_95_pp", 0 if val else None),
                     "relative_map50_95_pct": comparison.get("relative_map50_95_pct", 0 if val else None),
                     "actual_eligible_ratio": result.get("actual_eligible_channel_ratio") if result else 0,
                     "pruned_group_count": result.get("pruned_group_count") if result else 0,
                     "latency_median_ms": metric.get("latency", {}).get("median_ms"),
                     "structural_status": result["structural_smoke_status"] if result else "BASELINE",
                     "validation_status": result["validation_status"] if result else suite.get("baseline_validation_status")})
    return rows


def execute_suite(source_path, provenance, suite_dir, model_variant="n", suite_id=None):
    """Baseline dievaluasi sekali; setiap konfigurasi memuat sumber asli secara independen."""
    suite_dir = Path(suite_dir)
    suite_dir.mkdir(parents=True, exist_ok=True)
    (suite_dir / "baseline").mkdir(exist_ok=True)
    (suite_dir / "summary").mkdir(exist_ok=True)
    jobs = [{"policy": p, "ratio": r, "status": "NOT_RUN"} for p in policies_to_run(PRUNING_POLICY) for r in PRUNING_RATIOS]
    suite = {"format_version": 3, "model_variant": model_variant, "model_name": f"YOLO26{model_variant}",
             "suite_id": suite_id or suite_dir.name, "source": provenance,
             "configuration": configuration(model_variant=model_variant), "jobs": jobs, "results": [], "status": "RUNNING",
             "errors": [], "baseline_validation_status": "NOT_RUN",
             "v1_reference": {"source": "user_provided_brief", "verification": "UNVERIFIED",
                 "ratio": "UNKNOWN", "baseline_map50_95": 0.526105, "pruned_map50_95": 0.346850,
                 "baseline_gflops": 6.025830, "pruned_gflops": 5.840742,
                 "included_in_primary_comparison": False}}
    if model_variant != "n":
        suite["v1_reference"] = {"verification": "UNAVAILABLE", "reason": "No verified S V1 dataset metrics supplied",
                                  "included_in_primary_comparison": False, "ratio": "UNKNOWN"}
    write_json(suite_dir / "config.json", configuration(model_variant=model_variant))
    write_json(suite_dir / "checkpoint_provenance.json", provenance)
    write_json(suite_dir / "suite_manifest.json", suite)
    try:
        environment = environment_report()
        suite["environment"] = environment
        write_json(suite_dir / "environment.json", environment)
        baseline, dataset, base_info = _prepare_baseline(source_path, provenance, suite_dir, model_variant)
        suite.update({"baseline": baseline, "baseline_metadata": base_info,
                      "baseline_validation_status": base_info["validation_status"]})
        for job in jobs:
            job["status"] = "RUNNING"
            write_json(suite_dir / "suite_manifest.json", suite)
            folder = suite_dir / job["policy"] / f"pr{job['ratio']:g}"
            result = _run_configuration(source_path, provenance, folder, job["policy"], job["ratio"],
                                        baseline, dataset, base_info, environment, suite["suite_id"], model_variant)
            suite["results"].append(result)
            job["status"] = "PASS"
            write_json(suite_dir / "suite_manifest.json", suite)
            import gc
            gc.collect()
            if TARGET_DEVICE.type == "cuda":
                torch.cuda.empty_cache()
        suite["status"] = "COMPLETE"
    except Exception as error:
        suite["status"] = "FAIL"
        validation_report_path = suite_dir / "baseline" / "validation_report.json"
        if validation_report_path.is_file():
            suite["baseline_validation_status"] = json.loads(validation_report_path.read_text(encoding="utf-8"))["status"]
        for job in jobs:
            if job["status"] == "RUNNING":
                job["status"] = "FAIL"
        suite["errors"].append({"type": type(error).__name__, "message": safe_error(error)})
        raise
    finally:
        write_json(suite_dir / "suite_manifest.json", suite)
        rows = summary_rows(suite)
        write_json(suite_dir / "summary" / "metrics.json", rows)
        write_csv(suite_dir / "summary" / "metrics.csv", rows)
    return suite


def execute_models(suite_dir):
    """Orkestrasi N/S dengan baseline, provenance, hasil, dan failure report terpisah."""
    root = Path(suite_dir)
    root.mkdir(parents=True, exist_ok=True)
    (root / "comparison").mkdir(exist_ok=True)
    bundle = {"format_version": 3, "suite_id": root.name, "model_variants": list(MODEL_VARIANTS),
              "models": [], "jobs": [{"model_variant": variant, "status": "NOT_RUN"} for variant in MODEL_VARIANTS],
              "status": "RUNNING", "errors": [], "configuration": configuration()}
    write_json(root / "suite_manifest.json", bundle)
    try:
        baseline_names = None
        for job in bundle["jobs"]:
            variant = job["model_variant"]
            folder = root / f"yolo26{variant}"
            (folder / "baseline").mkdir(parents=True, exist_ok=False)
            job["status"] = "RUNNING"
            write_json(root / "suite_manifest.json", bundle)
            source, provenance = obtain_baseline(folder, variant)
            child = execute_suite(source, provenance, folder, variant, root.name)
            names = child["baseline_metadata"]["names"]
            if baseline_names is not None:
                assert_class_match(names, baseline_names)
            baseline_names = names
            plot_suite(child, folder / "summary")
            bundle["models"].append(child)
            job["status"] = "PASS"
            write_json(root / "suite_manifest.json", bundle)
        bundle["comparison_plots"] = plot_model_comparison(bundle["models"], root / "comparison")
        bundle["status"] = "COMPLETE"
    except Exception as error:
        bundle["status"] = "FAIL"
        for job in bundle["jobs"]:
            if job["status"] == "RUNNING":
                job["status"] = "FAIL"
        bundle["errors"].append({"type": type(error).__name__, "message": safe_error(error)})
        raise
    finally:
        write_json(root / "suite_manifest.json", bundle)
        rows = [row for child in bundle["models"] for row in summary_rows(child)]
        write_json(root / "comparison" / "metrics.json", rows)
        write_csv(root / "comparison" / "metrics.csv", rows)
    return bundle

In [ ]:
def plot_suite(suite, output_dir):
    """Hanya hasil aktual; panel akurasi kosong diberi NOT_RUN, tanpa angka pengganti."""
    import matplotlib.pyplot as plt
    output = Path(output_dir)
    output.mkdir(parents=True, exist_ok=True)
    rows = summary_rows(suite)
    plots = [
        ("ratio_vs_map50_95", "ratio", "map50_95", "Target eligible-channel ratio", "mAP50–95"),
        ("ratio_vs_parameter_reduction", "ratio", "parameter_reduction_pct", "Target eligible-channel ratio", "Parameter reduction (%)"),
        ("ratio_vs_gflops_reduction", "ratio", "gflops_reduction_pct", "Target eligible-channel ratio", "GFLOPs reduction (%)"),
        ("map_vs_gflops", "gflops", "map50_95", "GFLOPs", "mAP50–95"),
        ("map_vs_parameters", "parameters", "map50_95", "Parameters", "mAP50–95"),
    ]
    status = []
    for name, x, y, xlabel, ylabel in plots:
        fig, ax = plt.subplots(figsize=(7, 4.5))
        count = 0
        for policy in ["baseline", *policies_to_run(PRUNING_POLICY)]:
            values = [row for row in rows if row["policy"] == policy and row.get(x) is not None and row.get(y) is not None]
            if values:
                ax.plot([row[x] for row in values], [row[y] for row in values], "o-", label=policy)
                count += len(values)
        if not count:
            ax.text(0.5, 0.5, "NOT_RUN: validation V3 belum tersedia", ha="center", transform=ax.transAxes)
        else:
            ax.legend()
        ax.set(xlabel=xlabel, ylabel=ylabel)
        ax.grid(alpha=0.2)
        fig.tight_layout()
        fig.savefig(output / (name + ".png"), dpi=160)
        plt.close(fig)
        status.append({"plot": name, "status": "MEASURED" if count else "NOT_RUN"})
    fig, ax = plt.subplots(figsize=(10, 5))
    base = (suite.get("baseline", {}).get("validation") or {}).get("per_class", [])
    if base:
        ax.plot([r["name"] for r in base], [r["ap50_95"] for r in base], "o-", label="baseline")
        for result in suite["results"]:
            per_class = (result["pruned"].get("validation") or {}).get("per_class", [])
            if per_class:
                ax.plot([r["name"] for r in per_class], [r["ap50_95"] for r in per_class], "o-",
                        label=f"{result['policy']} pr{result['ratio']:g}")
        ax.legend(fontsize="small")
    else:
        ax.text(0.5, 0.5, "NOT_RUN: AP per kelas belum tersedia", ha="center", transform=ax.transAxes)
    ax.set(ylabel="AP50–95")
    ax.tick_params(axis="x", rotation=20)
    fig.tight_layout()
    fig.savefig(output / "per_class_ap.png", dpi=160)
    plt.close(fig)
    status.append({"plot": "per_class_ap", "status": "MEASURED" if base else "NOT_RUN"})
    write_json(output / "plot_status.json", status)
    return status

In [ ]:
"""Standalone ordinary fine-tuning helpers for native structurally pruned YOLO models.

The trainer retains the DetectionModel object instead of rebuilding its original YAML.
The detection criterion, gradients, optimizer, EMA, augmentations and validation remain
Ultralytics' native implementation. Only model loading, logging isolation and auditing
are customized. Best FP32 EMA is captured before SDK FP16 checkpoint serialization.
"""
import hashlib as _recovery_hashlib
import json as _recovery_json
from copy import deepcopy as _recovery_deepcopy
from datetime import UTC as _RECOVERY_UTC
from datetime import datetime as _RecoveryDateTime
from pathlib import Path as _RecoveryPath

import torch as _recovery_torch
from ultralytics import __version__ as _recovery_ultralytics_version
from ultralytics.models.yolo.detect import DetectionTrainer as _RecoveryDetectionTrainer
from ultralytics.nn.tasks import DetectionModel as _RecoveryDetectionModel


def recovery_ordered_names(names):
    """Preserve class order and reject sparse or non-integer class identifiers."""
    if isinstance(names, dict):
        if set(names) != set(range(len(names))):
            raise ValueError("Recovery class IDs must be contiguous integers starting at zero.")
        names = [names[index] for index in range(len(names))]
    if not isinstance(names, (list, tuple)) or not names or not all(isinstance(n, str) for n in names):
        raise ValueError("Recovery requires an explicit nonempty ordered class-name list.")
    return list(names)


def recovery_structure_signature(model):
    """Parameter shapes and physical Conv/BN dimensions; independent of learned values."""
    return {
        "parameters": {name: list(value.shape) for name, value in model.named_parameters()},
        "layers": {
            name: {"type": type(module).__name__, "in": module.in_channels,
                   "out": module.out_channels, "groups": module.groups}
            if isinstance(module, _recovery_torch.nn.Conv2d)
            else {"type": type(module).__name__, "features": module.num_features}
            for name, module in model.named_modules()
            if isinstance(module, (_recovery_torch.nn.Conv2d, _recovery_torch.nn.BatchNorm2d))
        },
    }


def recovery_state_sha256(model, *, parameters_only=False):
    digest = _recovery_hashlib.sha256()
    values = dict(model.named_parameters()) if parameters_only else model.state_dict()
    for name, value in sorted(values.items()):
        tensor = value.detach().cpu().contiguous()
        digest.update(name.encode())
        digest.update(str(tuple(tensor.shape)).encode())
        digest.update(str(tensor.dtype).encode())
        digest.update(tensor.numpy().tobytes())
    return digest.hexdigest()


def _recovery_write_json(path, value):
    path = _RecoveryPath(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(_recovery_json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")


def _recovery_native_snapshot(model):
    core = _recovery_deepcopy(model).to("cpu").float().eval()
    if type(core) is not _RecoveryDetectionModel:
        raise TypeError("Only an ordinary native DetectionModel can be serialized for recovery.")
    core.criterion = None
    return core


def _recovery_finite_state(model):
    return all(_recovery_torch.isfinite(v).all().item() for v in model.state_dict().values()
               if isinstance(v, _recovery_torch.Tensor) and v.is_floating_point())


class StructuralRecoveryTrainer(_RecoveryDetectionTrainer):
    """Native detection training with exact pruned-object loading and FP32 EMA snapshots."""

    def __init__(self, core, *, overrides, expected_names, source_sha256, epoch_callback=None):
        self.recovery_source_core = _recovery_native_snapshot(core)
        self.recovery_expected_names = recovery_ordered_names(expected_names)
        self.recovery_initial_signature = recovery_structure_signature(core)
        self.recovery_initial_state_sha256 = recovery_state_sha256(core)
        self.recovery_initial_parameters_sha256 = recovery_state_sha256(core, parameters_only=True)
        self.recovery_source_sha256 = source_sha256
        self.recovery_optimizer_steps = 0
        self.recovery_epochs = []
        self.recovery_best_core = None
        self.recovery_best_epoch = None
        self.recovery_initial_state_verified = False
        self.recovery_epoch_callback = epoch_callback
        self.recovery_filtered_callbacks = set()
        super().__init__(overrides=overrides)

    def run_callbacks(self, event):
        # BaseTrainer.__init__ dispatches integration events before returning. Filter
        # at dispatch time so W&B/Comet/Platform cannot initialize another run even
        # during construction. No global SETTINGS or callback registry is modified.
        for callback in self.callbacks.get(event, []):
            module = getattr(callback, "__module__", "")
            if module.startswith("ultralytics.utils.callbacks.") and module != "ultralytics.utils.callbacks.base":
                self.recovery_filtered_callbacks.add(module)
                continue
            callback(self)

    def setup_model(self):
        if self.resume or self.args.resume:
            raise ValueError("Recovery starts a new optimizer with resume=False.")
        if recovery_ordered_names(self.data["names"]) != self.recovery_expected_names:
            raise ValueError("Dataset class order differs from the pruned checkpoint.")
        self.model = self.recovery_source_core
        if recovery_ordered_names(self.model.names) != self.recovery_expected_names:
            raise ValueError("Checkpoint class order differs from recovery configuration.")
        self.model.criterion = None  # Native SDK builds its native criterion for this training run.
        self.model.requires_grad_(True)
        if recovery_structure_signature(self.model) != self.recovery_initial_signature:
            raise ValueError("Recovery loading changed the structurally pruned model.")
        if recovery_state_sha256(self.model) != self.recovery_initial_state_sha256:
            raise ValueError("Recovery loading changed the initial pruned weights.")
        self.recovery_initial_state_verified = True

    def get_model(self, cfg=None, weights=None, verbose=True):
        raise RuntimeError("Recovery must never rebuild the pruned model from its original YAML.")

    def optimizer_step(self):
        if not _recovery_torch.isfinite(self.loss).all().item():
            raise FloatingPointError("Recovery native detection loss became nonfinite.")
        if not all(_recovery_torch.isfinite(value).all().item() for value in self.loss_items.values()):
            raise FloatingPointError("Recovery native detection loss component became nonfinite.")
        super().optimizer_step()
        self.recovery_optimizer_steps += 1

    def _save_recovery_fp32(self, path, model):
        core = _recovery_native_snapshot(model)
        _recovery_torch.save({
            "model": core, "ema": None, "optimizer": None, "recovery_enabled": True,
            "epoch": int(self.epoch), "best_fitness": float(self.best_fitness),
            "train_args": vars(self.args).copy(), "version": _recovery_ultralytics_version,
            "date": _RecoveryDateTime.now(_RECOVERY_UTC).isoformat(),
            "recovery": {"format": "lamp-ordinary-recovery-fp32-v1",
                         "source_sha256": self.recovery_source_sha256,
                         "selection": "native validation fitness; EMA captured before SDK FP16 conversion"},
        }, path)
        return core

    def save_model(self):
        if recovery_structure_signature(self.model) != self.recovery_initial_signature:
            raise ValueError("Fine-tuning changed the pruned parameter shapes or Conv/BN dimensions.")
        if not _recovery_finite_state(self.model) or not _recovery_finite_state(self.ema.ema):
            raise FloatingPointError("Recovery model or EMA became nonfinite; no sanitized checkpoint is accepted.")
        if not _recovery_torch.isfinite(_recovery_torch.tensor(self.fitness)).item():
            raise FloatingPointError("Recovery validation fitness became nonfinite.")
        self.wdir.mkdir(parents=True, exist_ok=True)
        self._save_recovery_fp32(self.wdir / "last_fp32.pt", self.ema.ema)
        if self.best_fitness == self.fitness:
            self.recovery_best_core = self._save_recovery_fp32(self.wdir / "best_fp32.pt", self.ema.ema)
            self.recovery_best_epoch = int(self.epoch)
        row = {"epoch": int(self.epoch) + 1, "fitness": float(self.fitness),
               "losses": {k: float(v.detach().cpu()) for k, v in self.tloss.items()},
               "metrics": {k: float(v) for k, v in self.metrics.items()},
               "optimizer_steps": self.recovery_optimizer_steps,
               "learning_rates": [float(g["lr"]) for g in self.optimizer.param_groups]}
        self.recovery_epochs.append(row)
        _recovery_write_json(self.save_dir / "recovery_epoch_metrics.json", self.recovery_epochs)
        if self.recovery_epoch_callback is not None:
            self.recovery_epoch_callback(self, row)
        # Keep SDK best/last for native training diagnostics. The caller receives
        # the independently captured full-precision EMA, never these FP16 files.
        return super().save_model()


def recovery_training_arguments(checkpoint, data_yaml, output_dir, settings):
    """Explicit optimizer/protocol; callers can change the budget without changing method."""
    required = {"epochs", "optimizer", "lr0", "batch", "imgsz", "device", "seed"}
    if not required <= set(settings):
        raise ValueError(f"Missing recovery settings: {sorted(required - set(settings))}")
    if settings["optimizer"] == "auto":
        raise ValueError("Recovery requires an explicit optimizer so lr0 is not overridden.")
    if isinstance(settings["epochs"], bool) or not isinstance(settings["epochs"], int) or settings["epochs"] < 1:
        raise ValueError("Recovery epochs must be a positive integer.")
    arguments = {"model": str(checkpoint), "data": str(data_yaml), "project": str(_RecoveryPath(output_dir).parent),
                 "name": _RecoveryPath(output_dir).name, "exist_ok": False, "task": "detect", "mode": "train",
                 "resume": False, "pretrained": False, "amp": False, "quantize": None, "compile": False,
                 "distill_model": None, "freeze": None, "val": True, "save": True, "plots": True,
                 "patience": 0, "workers": 2, "cache": False, "deterministic": True, "channels_last": False}
    arguments.update(settings)
    for key, value in {"resume": False, "amp": False, "quantize": None, "compile": False,
                       "distill_model": None, "pretrained": False, "freeze": None}.items():
        if arguments[key] != value:
            raise ValueError(f"Ordinary FP32 recovery requires {key}={value!r}.")
    if not arguments["save"] or not arguments["val"]:
        raise ValueError("Recovery requires validation and checkpoint selection.")
    if not _RecoveryPath(checkpoint).is_file() or not _RecoveryPath(data_yaml).is_file():
        raise FileNotFoundError("Recovery checkpoint and dataset YAML must already exist locally.")
    return arguments


def fine_tune_pruned_checkpoint(checkpoint, data_yaml, output_dir, class_names, settings, epoch_callback=None):
    """Return (native best-EMA FP32 DetectionModel, report) after ordinary supervised fine-tuning."""
    output_dir = _RecoveryPath(output_dir)
    report_path = output_dir.parent / (output_dir.name + "_recovery_training_report.json")
    report = {"status": "RUNNING", "method": "ordinary-supervised-fine-tuning", "resume": False,
              "source_checkpoint": str(_RecoveryPath(checkpoint)), "errors": []}
    trainer = None
    try:
        arguments = recovery_training_arguments(checkpoint, data_yaml, output_dir, settings)
        source_sha256 = _recovery_hashlib.sha256(_RecoveryPath(checkpoint).read_bytes()).hexdigest()
        payload = _recovery_torch.load(checkpoint, map_location="cpu", weights_only=False)
        core = payload.get("ema") or payload.get("model")
        if type(core) is not _RecoveryDetectionModel:
            raise TypeError("Recovery input must contain a native structurally pruned DetectionModel.")
        if any(v.dtype != _recovery_torch.float32 for v in core.parameters() if v.is_floating_point()):
            raise ValueError("Recovery source must be saved in FP32; converting FP16 does not restore precision.")
        if recovery_ordered_names(core.names) != recovery_ordered_names(class_names):
            raise ValueError("Recovery source class order differs from requested classes.")
        report.update({"source_sha256": source_sha256, "arguments": arguments,
                       "initial_structure": recovery_structure_signature(core),
                       "initial_state_sha256": recovery_state_sha256(core)})
        _recovery_write_json(report_path, report)
        trainer = StructuralRecoveryTrainer(core, overrides=arguments, expected_names=class_names,
                                            source_sha256=source_sha256, epoch_callback=epoch_callback)
        trainer.train()
        if not trainer.recovery_initial_state_verified or trainer.recovery_optimizer_steps < 1:
            raise RuntimeError("Recovery did not verify the source or complete an optimizer step.")
        if trainer.recovery_best_core is None:
            raise RuntimeError("Recovery did not select a best validation EMA checkpoint.")
        if recovery_state_sha256(trainer.model, parameters_only=True) == trainer.recovery_initial_parameters_sha256:
            raise RuntimeError("Recovery completed without changing any trainable parameter.")
        recovered = trainer.recovery_best_core
        if recovery_structure_signature(recovered) != report["initial_structure"]:
            raise ValueError("Recovered model does not preserve the original pruned structure.")
        if not _recovery_finite_state(recovered):
            raise FloatingPointError("Recovered best EMA contains nonfinite weights.")
        report.update({"status": "PASS", "initial_state_verified": True, "structure_preserved": True,
                       "parameters_updated": True, "optimizer_steps": trainer.recovery_optimizer_steps,
                       "epochs_completed": len(trainer.recovery_epochs), "epochs": trainer.recovery_epochs,
                       "best_epoch": trainer.recovery_best_epoch + 1,
                       "best_selection": "native validation fitness; full-precision EMA snapshot",
                       "best_fitness": float(trainer.best_fitness),
                       "best_fp32_checkpoint": str(trainer.wdir / "best_fp32.pt"),
                       "last_fp32_checkpoint": str(trainer.wdir / "last_fp32.pt"),
                       "actual_save_dir": str(trainer.save_dir),
                       "filtered_sdk_integrations": sorted(trainer.recovery_filtered_callbacks),
                       "best_state_sha256": recovery_state_sha256(recovered)})
        _recovery_write_json(report_path, report)
        return recovered, report
    except Exception as error:
        report["status"] = "FAIL"
        report["errors"].append({"type": type(error).__name__, "message": str(error)})
        if trainer is not None:
            report.update({"optimizer_steps": trainer.recovery_optimizer_steps,
                           "epochs": trainer.recovery_epochs})
        _recovery_write_json(report_path, report)
        raise

In [ ]:
_run_pruning_configuration = _run_configuration
_pruning_summary_rows = summary_rows
_pruning_tracking_metrics = tracking_metrics


def run_identity(policy, ratio, suite_id, model_variant="n"):
    tag = f"pr{ratio:g}"
    return {"name": lamp_v2_display_name(policy, tag, model_variant, RECOVERY_EPOCHS),
            "id": uuid.uuid4().hex, "resume": "never", "group": f"lamp-finetune-{suite_id}",
            "job_type": "lamp-ordinary-finetune-recovery",
            "tags": [f"yolo26{model_variant}", "lamp", "structured", "pruned", policy, tag,
                     "finetune", "recovery", f"epochs{RECOVERY_EPOCHS}", f"seed{SEED}"]}


def track_result(result, run_dir):
    """The reused pruning stage defers tracking to the complete recovery experiment."""
    return {"status": "NOT_RUN", "uploads": [], "errors": [],
            "reason": "Tracking is owned by the pruning plus recovery experiment."}


def tracking_metrics(result):
    values = _pruning_tracking_metrics(result)
    values["recovery/completed"] = int(result.get("recovery_status") == "PASS")
    for key in ("parameters", "gflops", "checkpoint_bytes"):
        value = result.get("recovered", {}).get(key)
        if value is not None:
            values["recovered/" + key] = value
    for key, value in (result.get("recovered", {}).get("validation") or {}).items():
        if key in {"map50_95", "map50", "precision", "recall"} and value is not None:
            values["recovered/" + key] = value
    for comparison in ("recovered_vs_pruned", "recovered_vs_baseline"):
        for key in ("delta_map50_95_pp", "relative_map50_95_pct"):
            value = result.get(comparison, {}).get(key)
            if value is not None:
                values[comparison + "/" + key] = value
    return values


def _start_recovery_tracking(result, folder):
    report = {"status": "NOT_RUN", "uploads": [], "errors": []}
    run = None
    if not WANDB_TRACKING:
        report["reason"] = "WANDB_TRACKING=False"
        return None, report
    try:
        run = _init_tracking(result, folder)
        run.use_artifact(model_spec(result["model_variant"])["artifact"], type="model")
        report.update({"status": "RUNNING", "run_id": run.id, "run_url": run.url})
        return run, report
    except Exception as error:  # noqa: BLE001 -- Tracking is independent of local scientific results.
        report.update({"status": "FAIL", "errors": [{"message": safe_error(error)}]})
        if run is not None:
            try:
                run.finish(exit_code=1)
            except Exception as finish_error:  # noqa: BLE001 -- Do not leak a failed SDK run.
                report["errors"].append({"message": safe_error(finish_error)})
        return None, report


def _finish_recovery_tracking(result, folder, run, report):
    if run is None:
        write_json(folder / "tracking_report.json", report)
        return report
    try:
        run.log(tracking_metrics(result))
        run.summary.update({"recovery_status": result["recovery_status"],
                            "structural_smoke_status": result["structural_smoke_status"],
                            "recovery_validation_status": result["recovery_validation_status"]})
        if UPLOAD_WANDB_ARTIFACTS:
            tag = f"{result['ratio']:g}".replace(".", "p")
            stem = f"yolo26{result['model_variant']}-lamp-v2-{result['policy']}-finetune-pr{tag}-{result['suite_id']}"
            for label, kind, checkpoint in (("pruned", "model", folder / "pruned_fp32.pt"),
                                             ("recovered", "model", folder / "recovery/recovered_fp32.pt"),
                                             ("report", "report", None)):
                if checkpoint is not None and not checkpoint.is_file():
                    continue
                artifact = wandb.Artifact(stem + "-" + label, type=kind,
                    metadata={"source": result["provenance"], "stage": label, "seed": SEED,
                              "ratio": result["ratio"], "recovery_status": result["recovery_status"]})
                if checkpoint is not None:
                    artifact.add_file(str(checkpoint), name=checkpoint.relative_to(folder).as_posix())
                else:
                    for path in sorted(folder.rglob("*")):
                        relative = path.relative_to(folder)
                        if (path.is_file() and "wandb" not in relative.parts
                                and path.suffix in {".json", ".csv", ".md", ".png", ".svg", ".yaml"}):
                            artifact.add_file(str(path), name=relative.as_posix())
                logged = run.log_artifact(artifact)
                logged.wait(timeout=300)
                report["uploads"].append({"stage": label, "status": "CONFIRMED", "name": logged.qualified_name})
        if not report["errors"]:
            report["status"] = "PASS"
    except Exception as error:  # noqa: BLE001 -- Upload failures must preserve local outputs.
        report["status"] = "FAIL"
        report["errors"].append({"message": safe_error(error)})
    finally:
        try:
            run.finish(exit_code=0 if report["status"] == "PASS" and result["recovery_status"] != "FAIL" else 1)
        except Exception as error:  # noqa: BLE001 -- Persist SDK finish failures too.
            report["status"] = "FAIL"
            report["errors"].append({"message": safe_error(error)})
        write_json(folder / "tracking_report.json", report)
    return report


def _write_recovery_reports(folder, result):
    _write_run_reports(folder, result)
    write_json(folder / "before_after_metrics.json", {key: result.get(key) for key in (
        "baseline", "pruned", "recovered", "comparison", "recovered_vs_pruned", "recovered_vs_baseline")})
    write_json(folder / "recovery_manifest.json", result)
    write_json(folder / "recovery_status.json", {key: result.get(key) for key in (
        "recovery_status", "recovery_validation_status", "recovery_reload", "recovery_training", "errors")})


def save_recovered_fp32_checkpoint(core, path, provenance, training_report):
    """Use the same FP32 object format with explicit post-recovery metadata."""
    save_fp32_checkpoint(core, path, provenance)
    payload = torch.load(path, map_location="cpu", weights_only=False)
    payload.update({"recovery_enabled": True, "recovery_method": "ordinary_supervised_finetune",
                    "recovery_settings": recovery_settings(),
                    "recovery_epochs_completed": training_report["epochs_completed"],
                    "epoch": training_report["best_epoch"] - 1,
                    "best_fitness": training_report["best_fitness"],
                    "checkpoint_selection": "best_validation_fitness_ema_fp32"})
    torch.save(payload, path)
    return {"path": str(Path(path).resolve()), "sha256": sha256_file(path), "file_bytes": Path(path).stat().st_size}


def _run_configuration(source_path, provenance, folder, policy, ratio, baseline, dataset,
                       baseline_metadata, environment, suite_id, model_variant="n"):
    folder = Path(folder)
    run, tracking = None, {"status": "NOT_RUN", "uploads": [], "errors": []}
    result = None
    stage = "pruning"
    try:
        result = _run_pruning_configuration(source_path, provenance, folder, policy, ratio,
            baseline, dataset, baseline_metadata, environment, suite_id, model_variant)
        result.update({"recovery_status": "NOT_RUN", "recovery_validation_status": "NOT_RUN", "recovered": {}})
        result["stages"]["experiment"] = "RUNNING"
        recovery = folder / "recovery"
        recovery.mkdir(exist_ok=False)
        run, tracking = _start_recovery_tracking(result, folder)

        def log_epoch(_trainer, metrics):
            if run is None or tracking["status"] == "FAIL":
                return
            try:
                values = {"recovery_epoch/" + key: metrics[key] for key in ("epoch", "fitness", "optimizer_steps")}
                for key in ("losses", "metrics"):
                    values.update({f"recovery_epoch/{key}/{name}": value for name, value in metrics.get(key, {}).items()})
                run.log(values)
            except Exception as error:  # noqa: BLE001 -- A logger failure cannot abort fine-tuning.
                tracking["status"] = "FAIL"
                tracking["errors"].append({"message": safe_error(error)})

        if dataset is None:
            result["warnings"].append("Recovery NOT_RUN: dataset tidak tersedia; tidak ada training pengganti.")
            result["stages"]["experiment"] = "PARTIAL_NOT_RUN"
            write_json(recovery / "eda_status.json", {"status": "NOT_RUN", "reason": "dataset unavailable"})
            return result
        stage = "recovery_training"
        seed_environment(SEED)
        pruned_path = folder / "pruned_fp32.pt"
        pruned = YOLO(str(pruned_path), task="detect").model.float().eval()
        shapes = {name: list(value.shape) for name, value in pruned.state_dict().items()}
        core, training_report = fine_tune_pruned_checkpoint(
            pruned_path, dataset["yaml"], recovery / "training", dataset["class_names"],
            recovery_settings(), epoch_callback=log_epoch)
        result["recovery_training"] = training_report
        core = core.to(TARGET_DEVICE).float().eval()
        if {name: list(value.shape) for name, value in core.state_dict().items()} != shapes:
            raise ValueError("Struktur model hasil pruning berubah selama fine-tuning.")
        assert_class_match(core.names, baseline_metadata["names"])
        assert_layer_consistency(core)
        image = torch.zeros(1, 3, IMGSZ, IMGSZ, device=TARGET_DEVICE, dtype=torch.float32)
        forward_snapshot(core, image)
        measures = model_measurements(core, IMGSZ)
        if measures["parameters"] != result["pruned"]["parameters"]:
            raise ValueError("Jumlah parameter recovery tidak sama dengan model pruned.")
        stage = "recovery_save_reload"
        saved = save_recovered_fp32_checkpoint(core, recovery / "recovered_fp32.pt", {
            **provenance, "pruned_checkpoint_sha256": sha256_file(pruned_path),
            "recovery_method": "ordinary_supervised_finetune", "checkpoint_selection": "best_validation_fitness_ema_fp32"},
            training_report)
        result["saved_recovered"] = saved
        measures["checkpoint_bytes"] = saved["file_bytes"]
        result["recovery_reload"] = verify_fresh_reload(recovery / "recovered_fp32.pt", core, image, recovery,
                                                       timeout=RELOAD_TIMEOUT_SECONDS)
        write_json(recovery / "reload_report.json", result["recovery_reload"])
        result["recovered"] = measures
        stage = "recovery_validation"
        measures["validation"] = evaluate_checkpoint(recovery / "recovered_fp32.pt", dataset, recovery, "validation")
        result["recovery_validation_status"] = "PASS"
        stage = "recovery_latency"
        measures["latency"] = (cuda_latency(core, image, LATENCY_WARMUP, LATENCY_REPEATS)
            if RUN_LATENCY_BENCHMARK else {"status": "NOT_RUN", "reason": "disabled"})
        result["recovered_vs_baseline"] = metric_comparison(baseline, measures)
        result["recovered_vs_pruned"] = metric_comparison(result["pruned"], measures)
        result["recovery_status"] = "PASS"
        result["stages"]["recovery"] = "PASS"
        result["stages"]["experiment"] = "PASS"
        write_json(recovery / "eda_status.json", {
            "status": "MEASURED", "channel_structure": "UNCHANGED_FROM_PRUNED",
            "channel_eda": "../eda", "validation_exports": measures["validation"].get("exports"),
            "meaning": "Shape preservation and training completion do not guarantee recovered accuracy."})
        (recovery / "RESTORE.md").write_text(
            "# Model setelah ordinary fine-tuning\n\n"
            "```python\nfrom ultralytics import YOLO\nmodel = YOLO('recovered_fp32.pt', task='detect')\n"
            "model.predict('image.jpg', imgsz=640, nms=None, quantize=None)\n```\n\n"
            "Native DetectionModel struktural FP32; ema=None, optimizer=None. "
            "Subclass trainer tidak diperlukan untuk inference. Lihat environment.json pada direktori konfigurasi.\n",
            encoding="utf-8")
        return result
    except Exception as error:
        if result is None and (folder / "pruning_manifest.json").is_file():
            result = json.loads((folder / "pruning_manifest.json").read_text(encoding="utf-8"))
        if result is not None:
            result.setdefault("recovered", {})
            result["recovery_status"] = "FAIL" if stage != "pruning" else "NOT_RUN"
            result.setdefault("recovery_validation_status", "NOT_RUN")
            if stage == "recovery_validation":
                result["recovery_validation_status"] = "FAIL"
            result["stages"][stage] = "FAIL"
            result["stages"]["experiment"] = "FAIL"
            result["errors"].append({"stage": stage, "type": type(error).__name__, "message": safe_error(error)})
        raise
    finally:
        if result is not None:
            result["tracking"] = tracking
            _write_recovery_reports(folder, result)
            result["tracking"] = _finish_recovery_tracking(result, folder, run, tracking)
            _write_recovery_reports(folder, result)


def summary_rows(suite):
    rows = _pruning_summary_rows(suite)
    for row in rows:
        row["stage"] = "baseline" if row["policy"] == "baseline" else "pruned"
    for result in suite.get("results", []):
        if result.get("recovered"):
            entry = {**result, "pruned": result["recovered"], "comparison": result["recovered_vs_baseline"],
                     "validation_status": result["recovery_validation_status"]}
            row = _pruning_summary_rows({**suite, "results": [entry]})[-1]
            row.update({"stage": "recovered", "recovery_status": result["recovery_status"],
                        "recovery_gain_map50_95_pp": result["recovered_vs_pruned"].get("delta_map50_95_pp")})
            rows.append(row)
    return rows


def plot_suite(suite, output_dir):
    return plot_recovery_comparison([suite], output_dir)


def plot_model_comparison(suites, output_dir):
    return plot_recovery_comparison(suites, output_dir)


def plot_recovery_comparison(suites, output_dir):
    import matplotlib.pyplot as plt
    folder = Path(output_dir)
    folder.mkdir(parents=True, exist_ok=True)
    rows = [row for suite in suites for row in summary_rows(suite)]
    write_csv(folder / "three_stage_metrics.csv", rows)
    files = []
    for x, label in (("parameters", "Parameters"), ("gflops", "GFLOPs"), ("latency_median_ms", "Raw forward latency (ms)")):
        fig, ax = plt.subplots(figsize=(11, 7))
        count = 0
        for stage, marker in (("baseline", "s"), ("pruned", "x"), ("recovered", "o")):
            for row in rows:
                if row["stage"] != stage or row.get(x) is None or row.get("map50_95") is None:
                    continue
                ax.scatter(row[x], row["map50_95"], marker=marker,
                           color="tab:blue" if row["model_variant"] == "n" else "tab:orange")
                ax.annotate(f"{row['model']} {row['policy']} {stage} pr{row['ratio']:g}",
                            (row[x], row["map50_95"]), xytext=(4, 4), textcoords="offset points", fontsize=7)
                count += 1
        if not count:
            ax.text(.5, .5, "NOT_RUN: measured accuracy/latency unavailable", ha="center", transform=ax.transAxes)
        ax.set(xlabel=label, ylabel="mAP50–95", title="Baseline / pruned / ordinary fine-tuned")
        fig.tight_layout()
        files.append(_eda_save_figure(fig, folder / ("recovery_map_vs_" + x)))
    return files

In [ ]:
BUNDLE = execute_models(SUITE_DIR)
SUITES = BUNDLE["models"]
print("Suite:", BUNDLE["status"], "| model:", len(SUITES), "| eksperimen:", sum(len(s["results"]) for s in SUITES))

## 11 — EDA Identitas, Cakupan, dan Channel
Inventory Conv/BN, eligible/protected groups, indeks asli channel, skor LAMP finite,
dan perubahan per stage/region disimpan untuk setiap model/rasio. Diagram tiga panel:
**Before Pruning** biru, **After Pruning** merah, **Channel Comparison** overlay;
seluruh Conv-output memiliki urutan dan skala sumbu yang sama pada ketiga panel.
Overview serta detail maksimum 32 layer disimpan PNG 300 DPI, SVG, dan CSV sumber.
Heatmap menunjukkan channel bobot bertahan/terhapus, bukan aktivasi feature map.
Contoh detail default rasio 0.50; seluruh rasio tetap dianalisis dan disimpan.

In [ ]:
SUMMARY = pd.DataFrame([row for suite in SUITES for row in summary_rows(suite)])
display(SUMMARY[["model", "policy", "stage", "ratio", "actual_eligible_ratio", "pruned_group_count", "structural_status", "validation_status"]])
from IPython.display import Image as NotebookImage
for suite in SUITES:
    for result in suite["results"]:
        if abs(result["ratio"] - EDA_DISPLAY_RATIO) < 1e-12:
            print(result["model_name"], result["policy"], f"pr{result['ratio']:g}")
            overview = SUITE_DIR / result["model_name"].lower() / result["policy"] / f"pr{result['ratio']:g}" / "eda" / "channels_overview.png"
            if overview.is_file():
                display(NotebookImage(filename=str(overview)))
            display(pd.DataFrame(result["audit"]["eligible_groups"])[["name", "initial_channels", "group_id"]])

## 12 — EDA Efisiensi dan Metrik pada Tiga Tahap
Tabel membedakan `stage=baseline`, `pruned`, dan `recovered`. Parameter/GFLOPs/channel
setelah recovery harus sama dengan struktur pruned. Ukuran checkpoint memakai FP32 yang
sama; latency disertai interval p05–p95. Diagram channel dan ledger menjelaskan keputusan
pruning, sedangkan training curve menjelaskan jalannya fine-tuning.

Delta mAP dalam poin persentase dan perubahan relatif dicatat terpisah, termasuk gain
recovered versus pruned serta selisih recovered versus baseline. AP/P/R/F1/support per kelas,
kurva PR/F1/confidence, dan confusion matrix memakai hasil validator aktual. IoU pencocokan
confusion matrix dibedakan dari IoU konfigurasi validasi. Metrik yang belum diukur tetap null.

In [ ]:
display(SUMMARY[["model", "policy", "stage", "ratio", "parameters", "parameter_reduction_pct", "gflops", "gflops_reduction_pct", "checkpoint_bytes", "latency_median_ms"]])
display(SUMMARY[["model", "policy", "stage", "ratio", "map50_95", "map50", "precision", "recall", "delta_map50_95_pp", "relative_map50_95_pct", "validation_status"]])
PER_CLASS = []
for suite in SUITES:
    PER_CLASS.extend({"model": suite["model_name"], "policy": "baseline", "stage": "baseline", "ratio": 0, **row}
                     for row in (suite["baseline"].get("validation") or {}).get("per_class", []))
    for result in suite["results"]:
        for stage in ("pruned", "recovered"):
            PER_CLASS.extend({"model": suite["model_name"], "policy": result["policy"], "stage": stage,
                              "ratio": result["ratio"], **row}
                             for row in (result.get(stage, {}).get("validation") or {}).get("per_class", []))
write_json(SUITE_DIR / "comparison" / "per_class_metrics.json", PER_CLASS)
write_csv(SUITE_DIR / "comparison" / "per_class_metrics.csv", PER_CLASS)

## 13 — Trade-off Kompresi dan Pemulihan Akurasi
Grafik memakai hasil aktual dengan identitas model, rasio, policy, dan tahap sebelum/sesudah
recovery. Metrik yang tidak dijalankan tetap NOT_RUN. Fine-tuning dapat memperbaiki akurasi,
tetapi pemulihan penuh tidak dijanjikan, terutama pada rasio agresif.
Seed tunggal tidak membuktikan generalisasi statistik; perubahan channel/akurasi tidak
otomatis membuktikan layer tertentu sebagai penyebab. Hasil notebook tanpa recovery
yang lama tetap bukti historis dan tidak dicampur sebagai konfigurasi suite ini.

In [ ]:
print("Perbandingan model:")
display(BUNDLE["comparison_plots"])
for suite in SUITES:
    print(suite["model_name"], "| validasi:", suite["baseline_validation_status"])
    display(pd.DataFrame([{"ratio": r["ratio"], "policy": r["policy"], "eda_status": r["eda_status"]} for r in suite["results"]]))

## 14 — Artifact, Status, dan Reproducibility
Project W&B tetap `yolom26-kd-indotraffic`; setiap model/policy/rasio menggunakan UUID/run
baru dengan label fine-tuning recovery dan `resume="never"`. Baseline, pruned, serta recovered
memiliki metrik terpisah. Tracking/upload yang gagal tidak menghapus hasil lokal dan tidak
dianggap berhasil. Kredensial tidak disimpan dalam config, artifact, atau laporan.

Output suite baru berada di `pruning/lamp_v2_finetune/{suite_id}/`, berisi direktori `yolo26n/`,
`yolo26s/`, dan `comparison/`. Simpan checkpoint pruned, `recovery/recovered_fp32.pt`,
best/last training, audit channel/skor, EDA, config, provenance, environment, log epoch,
manifest recovery, reload/prediction report, serta metrik ketiga tahap.
Notebook lama dan laporan historis tidak ditulis ulang. Structural validity, training
completion, evaluation validity, tracking status, dan scientific trade-off tetap terpisah.

In [ ]:
write_json(SUITE_DIR / "comparison" / "reproducibility.json", {
    "suite_id": BUNDLE["suite_id"], "configuration": configuration(),
    "sources": [{"model": s["model_name"], "source": s["source"]} for s in SUITES],
    "baseline_validation_statuses": {s["model_name"]: s["baseline_validation_status"] for s in SUITES},
    "structural_statuses": [{"model": s["model_name"], "policy": r["policy"], "ratio": r["ratio"], "status": r["structural_smoke_status"]} for s in SUITES for r in s["results"]],
    "recovery_statuses": [{"model": s["model_name"], "policy": r["policy"], "ratio": r["ratio"], "status": r.get("recovery_status", "NOT_RUN")} for s in SUITES for r in s["results"]],
    "recovery_validation_statuses": [{"model": s["model_name"], "policy": r["policy"], "ratio": r["ratio"], "status": r.get("recovery_validation_status", "NOT_RUN")} for s in SUITES for r in s["results"]],
    "tracking_statuses": [{"model": s["model_name"], "policy": r["policy"], "ratio": r["ratio"], **r["tracking"]} for s in SUITES for r in s["results"]],
    "plots": BUNDLE["comparison_plots"], "limitations": ["Single seed", "GPU/mAP only when actually measured", "Descriptive associations, no causal layer attribution"]
})
print("Output:", SUITE_DIR)
display(SUMMARY)